# Development and Internal Evaluation of Seven Prespecified Classifiers for Self-Reported Diagnosed Diabetes Status in BRFSS 2025

> **Study type:** cross-sectional methodological benchmark with internal validation.  
> **Not claimed:** prospective diabetes-risk prediction, laboratory diagnosis, external validation, causal inference, or a deployable clinical screening tool.

| Study item | Frozen design |
|---|---|
| Source | CDC BRFSS 2025 public-use file |
| Outcome | `DIABETE4=1` vs `DIABETE4=3` |
| Eligible cohort | 342,539 respondents |
| Outcome-positive respondents | 51,827 (15.13% unweighted sample proportion) |
| Predictors | 20 literature-grounded survey variables |
| Split | 80/20 stratified, seed 42 |
| CV | Stratified 5-fold on training only |
| Primary resampling | None |
| Models | LR, DT, RF, GB, XGBoost, Linear SVM, Gaussian NB |
| Final evaluation | Prespecified metrics on a held-out internal test set |
| Model comparison | Paired respondent bootstrap + Holm-corrected McNemar |

### Metric terminology

The historical machine-readable field `pr_auc` is retained for artifact compatibility, but it is computed with scikit-learn `average_precision_score`. Therefore its correct human-facing name is **Average Precision (AP)**, not trapezoidal area under the precision-recall curve.

## One-notebook rule

This notebook remains the **single executable source of truth** for the frozen primary study: dataset rationale → data preparation → seven prespecified model configurations → paired comparison → interpretation.


# Part I — Dataset, Target, and Predictor Design

## 1.1 Why BRFSS 2025?

| Reason | Why it matters |
|---|---|
| Large public health survey | Supports stable internal benchmarking on a large sample |
| `DIABETE4` outcome | Direct self-report of diagnosed diabetes status |
| Broad predictor domains | Demographic, socioeconomic, behavioral, access, and health indicators |
| Official documentation | CDC codebook, calculated-variable definitions, and complex-survey documentation |
| Recent release | Contemporary 2025 cross-sectional snapshot |

### Participant flow

| Stage | n |
|---|---:|
| Raw BRFSS 2025 records | 356,158 |
| Excluded: prediabetes/borderline (`DIABETE4=4`) | 10,166 |
| Excluded: pregnancy-only diabetes (`DIABETE4=2`) | 2,671 |
| Excluded: don't know (`7`) | 580 |
| Excluded: refused (`9`) | 196 |
| Excluded: missing target | 6 |
| **Primary binary cohort (`1` or `3`)** | **342,539** |
| Training partition | 274,031 |
| Held-out internal test partition | 68,508 |

### Intended use and limits

- **Intended:** reproducible comparison of seven prespecified classifier configurations on the same BRFSS 2025 cohort and internal holdout.
- **Not intended:** population-prevalence estimation, a clinical screening rule, individual treatment decisions, causal inference, or prospective incidence prediction.
- Outcome is **self-reported diagnosed diabetes**, not laboratory-confirmed disease.
- Predictor/outcome timing is cross-sectional; temporal order is generally unknown.
- The benchmark is unweighted and therefore estimates **sample-level predictive performance**.
- The random holdout is **internal validation only**.


## 1.2 Literature anchors — what is reused and what is not

| ID | Data / target | Key protocol | How this study uses it |
|---|---|---|---|
| **P01** Nayem & Biswas (2026) | Raw BRFSS 2014–2024; binary diabetes | Logistic regression; complete-case primary analysis; age ≥40 | Predictor-selection precedent and large-scale BRFSS context |
| **P02** Xie et al. (2019) | Raw BRFSS 2014; type-2-oriented cohort | 2/3 train, 1/3 test; complete cases; SMOTE on train only | Model/feature precedent; leakage-safe resampling precedent |
| **P03** Muhammad et al. (2025) | Tennessee BRFSS 2023; self-reported diabetes | 80/20 stratified; MICE; train-only SMOTE; 5-fold CV; model tuning | Closest methodological comparator for metrics and model comparison |
| **P05** Li et al. (2024) | BRFSS 2021 | XGBoost / GA-XGBoost; resampling and ensembles | XGBoost precedent only; not a numerical target |
| **P06** Aich et al. (2026) | BRFSS-2015-derived UCI dataset; prediabetes+diabetes positive | Feature-selection benchmark | Indirect feature-selection evidence only |

**Important:** our pipeline is **not an exact reproduction** of any paper. Different cohorts, targets, missing-data handling, resampling, feature sets, and tuning make direct metric matching inappropriate.

Core DOIs:  
P01 `10.1038/s41598-026-46927-7` · P02 `10.5888/pcd16.190109` · P03 `10.1177/21501319251400546` · P05 `10.1371/journal.pone.0311222` · P06 `10.1038/s41598-026-41874-9`


# II. Initial Exploration of BRFSS 2025

## 2.1 Data Source and File Integrity

The source file used here is the official CDC 2025 SAS Transport dataset (`.XPT`). The GitHub/Kaggle pipeline verifies the payload before analysis.

The original CDC ZIP is preserved separately. Kaggle uses a normalized archive member name because the CDC ZIP contains an XPT filename with a trailing space.



In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import pandas as pd
from IPython.display import display

TARGET = "DIABETE4"
EXPECTED_XPT_BYTES = 799_971_280
EXPECTED_XPT_SHA256 = "d99262f8854f018a600c3538ee78e4308569af700abe6b92b36180336c24dec2"

xpt_candidates = list(Path("/kaggle/input").rglob("LLCP2025.XPT"))
zip_candidates = list(Path("/kaggle/input").rglob("LLCP2025XPT.zip"))

print("XPT candidates:", [str(p) for p in xpt_candidates])
print("ZIP candidates:", [str(p) for p in zip_candidates])

if len(xpt_candidates) == 1:
    xpt_path = xpt_candidates[0]
    source_mode = "Kaggle dataset extracted XPT"
elif len(xpt_candidates) == 0 and len(zip_candidates) == 1:
    zip_path = zip_candidates[0]
    working_dir = Path("/kaggle/working/brfss_2025")
    working_dir.mkdir(parents=True, exist_ok=True)

    with ZipFile(zip_path) as zf:
        members = [info for info in zf.infolist() if not info.is_dir()]
        if len(members) != 1:
            raise RuntimeError(f"Expected one data payload, found {len(members)}.")
        member = members[0]
        normalized_name = Path(member.filename.strip()).name
        if normalized_name != "LLCP2025.XPT":
            raise RuntimeError(f"Unexpected payload name: {normalized_name!r}")
        xpt_path = working_dir / normalized_name
        with zf.open(member) as src, xpt_path.open("wb") as dst:
            for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                dst.write(chunk)
    source_mode = "Kaggle dataset ZIP fallback"
else:
    raise RuntimeError(
        "Expected exactly one BRFSS input payload from Kaggle Dataset. "
        f"Found {len(xpt_candidates)} XPT and {len(zip_candidates)} ZIP candidates."
    )

actual_bytes = xpt_path.stat().st_size
print("Source mode:", source_mode)
print("XPT path:", xpt_path)
print("XPT bytes:", f"{actual_bytes:,}")

if actual_bytes != EXPECTED_XPT_BYTES:
    raise RuntimeError(
        f"Unexpected XPT size: {actual_bytes:,}; expected {EXPECTED_XPT_BYTES:,}."
    )

sha256 = hashlib.sha256()
with xpt_path.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)
xpt_sha256 = sha256.hexdigest()
print("XPT SHA-256:", xpt_sha256)

if xpt_sha256 != EXPECTED_XPT_SHA256:
    raise RuntimeError(
        f"Unexpected XPT SHA-256: {xpt_sha256}; expected {EXPECTED_XPT_SHA256}."
    )


In [ ]:
print("Validated Kaggle input payload.")
print("Ready to load SAS Transport file:", xpt_path)


## 2.2 Load and Inspect the Dataset

We first verify the shape, memory footprint, target availability, and a small sample of rows before performing any analytical recoding.



In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

print("Shape:", df.shape)
print("Rows:", f"{len(df):,}")
CDC_DOCUMENTED_VARIABLES = 284
print("Columns loaded by pandas:", df.shape[1])
print("CDC-documented XPT variables:", CDC_DOCUMENTED_VARIABLES)
print("Column-count delta:", df.shape[1] - CDC_DOCUMENTED_VARIABLES)
print("Memory usage:", f"{df.memory_usage(deep=True).sum() / (1024**3):.3f} GiB")

if TARGET not in df.columns:
    raise RuntimeError(f"Target {TARGET!r} not found in dataset.")

if df.shape[1] != CDC_DOCUMENTED_VARIABLES:
    print(
        "AUDIT NOTE: CDC documents 284 XPT variables, while pandas exposes "
        f"{df.shape[1]}. Critical study columns are checked explicitly below; "
        "the one-column schema discrepancy remains a parser/schema audit item."
    )

display(df.head())


## 1.3 Target definition — `DIABETE4`

| Raw code | Primary analysis |
|---:|---|
| 1 | Diabetes = 1 |
| 3 | No diabetes = 0 |
| 2 | Exclude: pregnancy-only diabetes |
| 4 | Exclude: prediabetes / borderline |
| 7 / 9 / missing | Exclude: unknown / refused / missing |

**Interpretation:** this is diagnosed-diabetes **status classification**. It does not distinguish Type 1 from Type 2 and must not be described as future diabetes incidence prediction.


In [ ]:
target_counts = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis(TARGET)
    .reset_index(name="count")
)
target_counts["percent"] = target_counts["count"] / len(df) * 100

display(target_counts)
target_counts.to_csv("/kaggle/working/target_distribution.csv", index=False)

primary_target = df[TARGET].isin([1.0, 3.0])
n_primary = int(primary_target.sum())
n_diabetes = int((df[TARGET] == 1.0).sum())
n_no_diabetes = int((df[TARGET] == 3.0).sum())

print(f"Primary binary cohort: {n_primary:,}")
print(f"Diabetes: {n_diabetes:,}")
print(f"No diabetes: {n_no_diabetes:,}")
print(f"Diabetes prevalence in binary cohort: {n_diabetes / n_primary * 100:.2f}%")


## 2.4 Missingness and Variable Coverage

BRFSS uses two kinds of missingness:

1. native SAS missing values, which appear as `NaN`;
2. explicit response codes such as `7`, `9`, `77`, or `99` for don't know/refused/not sure, depending on the variable.

Therefore, raw `NaN` percentages are useful for inspection but are **not yet analytical missingness**. Coding-specific missing values are handled in the variable coding audit before preprocessing.



In [ ]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100),
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
}).sort_values(["missing_pct", "column"], ascending=[False, True])

print("Columns with the highest native SAS missingness:")
display(column_summary.head(30))

column_summary.to_csv("/kaggle/working/column_summary.csv", index=False)


## 1.4 Predictor-selection rules

A predictor is retained only if it passes all five checks:

1. literature support in BRFSS/diabetes work;
2. usable analytical coverage after special-code recoding;
3. no direct diabetes-diagnosis/treatment leakage;
4. limited redundancy with selected variables;
5. useful and interpretable domain coverage.

**No supervised feature selection is run on the held-out test set.**


### Hard leakage exclusions

These diabetes-specific fields are prohibited because they directly reveal diagnosis history, treatment, monitoring, or diabetes-specific care:

`DIABAGE4`, `INSULIN1`, `CHKHEMO3`, `EYEEXAM1`, `DIABEYE1`, `DIABEDU1`.

### Temporal-ambiguity note

Selected concurrent health variables such as hypertension, high cholesterol, coronary disease, stroke, kidney disease, walking difficulty, and healthcare access are **not treated as direct target leakage** for cross-sectional classification. However, their temporal relationship to diabetes diagnosis is unknown.

Therefore they may support **status classification**, but they must not be interpreted as causal risk factors or prospective predictors.


## 3.3 Final Primary Predictor Set

The primary predictor set is frozen at **20 variables**.

It intentionally covers demographics, socioeconomic status, anthropometrics, general/mental health, lifestyle, cardiometabolic health, major comorbidities, functional health, and healthcare access while avoiding unnecessary duplication.



In [ ]:
PRIMARY_FEATURES = [
    "_AGEG5YR",
    "_SEX",
    "_RACEGR3",
    "_EDUCAG",
    "_INCOMG1",
    "EMPLOY1",
    "_BMI5",
    "GENHLTH",
    "PHYSHLTH",
    "MENTHLTH",
    "_TOTINDA",
    "_RFSMOK3",
    "SLEPTIM1",
    "_RFHYPE6",
    "_RFCHOL3",
    "_MICHD",
    "CVDSTRK3",
    "CHCKDNY2",
    "DIFFWALK",
    "PERSDOC3"
]

feature_domains = {
    "_AGEG5YR": "Demographic",
    "_SEX": "Demographic",
    "_RACEGR3": "Demographic",
    "_EDUCAG": "Socioeconomic",
    "_INCOMG1": "Socioeconomic",
    "EMPLOY1": "Socioeconomic",
    "_BMI5": "Anthropometric",
    "GENHLTH": "General health",
    "PHYSHLTH": "General health",
    "MENTHLTH": "Mental health",
    "_TOTINDA": "Lifestyle",
    "_RFSMOK3": "Lifestyle",
    "SLEPTIM1": "Lifestyle",
    "_RFHYPE6": "Cardiometabolic",
    "_RFCHOL3": "Cardiometabolic",
    "_MICHD": "Comorbidity",
    "CVDSTRK3": "Comorbidity",
    "CHCKDNY2": "Comorbidity",
    "DIFFWALK": "Functional health",
    "PERSDOC3": "Healthcare access",
}

missing_features = [c for c in PRIMARY_FEATURES if c not in df.columns]
if missing_features:
    raise RuntimeError(f"Frozen predictors missing from BRFSS 2025: {missing_features}")

feature_audit = pd.DataFrame({
    "variable": PRIMARY_FEATURES,
    "domain": [feature_domains[c] for c in PRIMARY_FEATURES],
    "native_missing_pct": [df[c].isna().mean() * 100 for c in PRIMARY_FEATURES],
    "n_unique_nonnull": [df[c].nunique(dropna=True) for c in PRIMARY_FEATURES],
})

display(feature_audit)
feature_audit.to_csv("/kaggle/working/final_feature_audit.csv", index=False)


## 3.4 Why Some Plausible Variables Were Not Selected

Examples of deliberate exclusions:

- `_BMI5CAT` — redundant with continuous `_BMI5`.
- `_RFHLTH` — derived from `GENHLTH`.
- `MARITAL` — lower priority after age, sex, education, income, employment, and race/ethnicity are represented.
- `_HLTHPL2`, `MEDCOST1`, `CHECKUP1` — useful access/use variables, but only `PERSDOC3` is retained in the compact primary set.
- `DRNKANY6`, `_RFBING6` — valid lifestyle variables, but activity, smoking, and sleep already provide interpretable behavioral coverage.
- `ASTHMA3`, `HAVARTH4` — weaker diabetes-specific rationale than the selected cardiovascular, kidney, and functional-health indicators.
- `ADDEPEV3` — overlaps the broader `MENTHLTH` construct.

These variables remain available for sensitivity analyses but are not part of the primary benchmark.



## 3.5 Interpretation Note

`_RACEGR3` is retained as a population-demographic covariate because prior raw-BRFSS studies include race/ethnicity and because diabetes prevalence can differ across population groups.

Any association must **not** be interpreted as a biological causal effect. Observed differences may reflect social, environmental, healthcare-access, socioeconomic, and structural factors not fully represented by the remaining variables.



## 1.5 Part I freeze point

- Official BRFSS 2025 payload verified by byte size **and SHA-256**.
- Binary target frozen: `DIABETE4=1` vs `3`.
- Hard diabetes-specific leakage variables excluded.
- Primary predictor set frozen at **20 variables**.
- Remaining 283-vs-284 schema-count discrepancy is documented and does not silently alter the target or selected predictors.


In [ ]:
overview = {
    "source_mode": source_mode,
    "source_file": xpt_path.name,
    "xpt_sha256": xpt_sha256,
    "xpt_bytes": int(actual_bytes),
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "target": TARGET,
    "target_missing": int(df[TARGET].isna().sum()),
    "memory_gib": round(float(df.memory_usage(deep=True).sum() / (1024**3)), 3),
}

with open("/kaggle/working/dataset_overview.json", "w", encoding="utf-8") as f:
    json.dump(overview, f, indent=2)

print(json.dumps(overview, indent=2))
print("Initial exploration complete.")


# Part II — Leakage-Safe Data Preparation

| Step | Rule |
|---|---|
| Target filtering | Build binary cohort before modeling |
| Special codes | Convert BRFSS non-response codes to missing |
| Split | 80/20 stratified before learned preprocessing |
| Categorical missing | Explicit missing category |
| Numeric missing | Training-only median imputation |
| Encoding | One-hot, unknown categories ignored |
| Scaling | Training-only StandardScaler for continuous fields |
| CV | Stratified 5-fold inside training only |
| Resampling | None in the primary benchmark |
| Test set | Held out from fitting/tuning; used for prespecified final evaluation |

### Sample-size rationale

This is a secondary analysis of the **full eligible public-use cohort**, not a prospectively recruited model-development sample. No arbitrary down-sampling was used. The development set contains 41,462 outcome-positive respondents for 20 raw predictors (82 transformed columns), so sparse-event instability is not the limiting issue. No formal prospective prediction-model sample-size calculation was performed; this is reported as a limitation rather than retrofitted after analysis.

### Survey estimand

BRFSS is a complex survey. CDC provides `_LLCPWT`, `_STSTR`, and `_PSU` for design-based population analyses. These variables are **not predictors** in the primary benchmark.

The primary estimand here is unweighted predictive performance in the analyzed respondent sample. Any claim about U.S. population prevalence or population-representative predictive performance requires a separately specified survey-weighted analysis.


## 2.2 Construct the Binary Target and Recode the 20 Predictors

The primary outcome compares respondents reporting diagnosed diabetes (`DIABETE4=1`) with respondents reporting no diabetes (`DIABETE4=3`).

Gestational diabetes, prediabetes/borderline diabetes, don't-know/refused responses, and missing target responses are excluded from the primary binary cohort.

For predictors, CDC non-response codes are converted to analytical missing values. Examples:

- `_AGEG5YR=14` → missing;
- `_EDUCAG=9` → missing;
- `_INCOMG1=9` → missing;
- `PHYSHLTH/MENTHLTH=88` → zero days, while 77/99 → missing;
- `SLEPTIM1=1..24` → valid hours, while 77/99 → missing;
- calculated binary variables with code 9 → missing.

The continuous BMI field `_BMI5` is stored as BMI × 100 and is divided by 100.



In [ ]:
import numpy as np

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)

    # Categorical / ordinal survey variables
    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])

    # Continuous variables
    out["_BMI5"] = raw["_BMI5"] / 100.0

    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)

    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)

    return out[PRIMARY_FEATURES]

target_mask = df["DIABETE4"].isin([1.0, 3.0])
analysis_raw = df.loc[target_mask, PRIMARY_FEATURES].copy()
X = recode_brfss_2025(analysis_raw)
y = (df.loc[target_mask, "DIABETE4"] == 1.0).astype("int8").rename("diabetes")

assert len(X) == len(y)
assert X.columns.tolist() == PRIMARY_FEATURES
assert set(y.unique()) == {0, 1}

print("Analysis cohort:", f"{len(X):,}")
print("Diabetes:", f"{int(y.sum()):,}")
print("No diabetes:", f"{int((y == 0).sum()):,}")
print("Sample diabetes prevalence:", f"{y.mean() * 100:.2f}%")
display(X.head())


## 2.3 Analytical Missingness After Recoding

Analytical missingness is recomputed **after** special-value decoding. This is the missingness relevant to preprocessing.

We also quantify how many respondents would remain under a strict complete-case strategy. This provides an empirical reason for choosing imputation rather than automatically discarding every row with at least one missing predictor.



In [ ]:
analytical_missingness = pd.DataFrame({
    "variable": PRIMARY_FEATURES,
    "domain": [feature_domains[c] for c in PRIMARY_FEATURES],
    "missing_count": [int(X[c].isna().sum()) for c in PRIMARY_FEATURES],
    "missing_pct": [float(X[c].isna().mean() * 100) for c in PRIMARY_FEATURES],
    "n_unique_nonnull": [int(X[c].nunique(dropna=True)) for c in PRIMARY_FEATURES],
}).sort_values("missing_pct", ascending=False)

complete_case_mask = X.notna().all(axis=1)
complete_case_n = int(complete_case_mask.sum())
complete_case_pct = complete_case_n / len(X) * 100

print(f"Rows in primary cohort: {len(X):,}")
print(f"Complete cases across all 20 predictors: {complete_case_n:,} ({complete_case_pct:.2f}%)")
print(f"Rows lost by complete-case deletion: {len(X)-complete_case_n:,} ({100-complete_case_pct:.2f}%)")

display(analytical_missingness)
analytical_missingness.to_csv("/kaggle/working/analytical_missingness.csv", index=False)


## 2.4 Missing-Data Strategy

The primary pipeline **retains the full binary target cohort** and handles missing predictors inside the training pipeline:

- **Categorical predictors:** missing values become an explicit `-1` category, then one-hot encoded.
- **Continuous predictors:** missing values are median-imputed using the training data only.
- **Continuous scaling:** standardized using training-set mean and standard deviation.

Why not complete-case deletion?

- income alone has substantial non-response;
- BMI, cholesterol, smoking, and sleep also contribute missingness;
- deleting every row with any missing predictor can remove a large fraction of respondents and alter the analyzed sample.

Why an explicit category for categorical missingness?

It avoids inventing a substantive category such as the modal income group for a respondent who refused or did not report income. This choice is transparent and can later be tested in sensitivity analysis.



## 2.5 Train/test split

- **80% train:** preprocessing fitting, 5-fold CV, and final model fitting.
- **20% internal test:** held out from preprocessing/model development.
- **Stratified:** preserves the sample class ratio.
- **Seed:** 42.
- **Primary benchmark:** no hyperparameter search and no threshold optimization.

Because BRFSS is cross-sectional, this is a respondent-level **internal random holdout**, not temporal or external validation.


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

RANDOM_STATE = 42
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame([
    {
        "split": "full",
        "n": len(y),
        "diabetes_n": int(y.sum()),
        "no_diabetes_n": int((y == 0).sum()),
        "diabetes_pct": float(y.mean() * 100),
    },
    {
        "split": "train",
        "n": len(y_train),
        "diabetes_n": int(y_train.sum()),
        "no_diabetes_n": int((y_train == 0).sum()),
        "diabetes_pct": float(y_train.mean() * 100),
    },
    {
        "split": "test",
        "n": len(y_test),
        "diabetes_n": int(y_test.sum()),
        "no_diabetes_n": int((y_test == 0).sum()),
        "diabetes_pct": float(y_test.mean() * 100),
    },
])

display(split_summary)
split_summary.to_csv("/kaggle/working/split_summary.csv", index=False)

assert set(X_train.index).isdisjoint(set(X_test.index))
assert len(X_train) + len(X_test) == len(X)


## 2.6 Leakage-Safe Preprocessing Pipeline

A single reusable `ColumnTransformer` is defined for the baseline model comparison.

Categorical variables are imputed with an explicit missing category and one-hot encoded. Numerical variables are median-imputed and standardized.

The transformer is **fit on `X_train` only**. Calling `fit_transform` on the full dataset before the split would leak information from the test set.



In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ],
    remainder="drop",
)

preprocessor_audit = clone(preprocessor)
X_train_ready = preprocessor_audit.fit_transform(X_train)
X_test_ready = preprocessor_audit.transform(X_test)

feature_names = preprocessor_audit.get_feature_names_out()

print("Training matrix:", X_train_ready.shape)
print("Test matrix:", X_test_ready.shape)
print("Transformed feature count:", len(feature_names))
print("First transformed feature names:")
print(feature_names[:25])

assert X_train_ready.shape[0] == len(X_train)
assert X_test_ready.shape[0] == len(X_test)
assert X_train_ready.shape[1] == X_test_ready.shape[1]


## 2.7 Stratified 5-fold cross-validation

CV is used **inside the training set** to describe stability across folds.

- preprocessing remains inside each model pipeline;
- every fold fits its own imputers/encoder/scaler;
- reported CV uncertainty is **mean + SD + range**, not a naive 95% t-interval;
- held-out uncertainty is estimated with respondent bootstrap intervals; these intervals are estimation aids, not multiplicity-adjusted confirmatory tests.

Reason: K-fold scores are correlated because training sets overlap, and stratification can further shrink apparent inter-fold variability.


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_rows = []
for fold, (fit_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
    y_fit = y_train.iloc[fit_idx]
    y_val = y_train.iloc[val_idx]
    fold_rows.append({
        "fold": fold,
        "fit_n": len(fit_idx),
        "validation_n": len(val_idx),
        "fit_diabetes_pct": float(y_fit.mean() * 100),
        "validation_diabetes_pct": float(y_val.mean() * 100),
    })

cv_fold_summary = pd.DataFrame(fold_rows)
display(cv_fold_summary)
cv_fold_summary.to_csv("/kaggle/working/cv_fold_summary.csv", index=False)


## 2.8 Frozen Part II design

| Component | Frozen choice |
|---|---|
| Target | `DIABETE4=1` vs `3` |
| Predictors | 20 literature-grounded variables |
| Split | 80/20 stratified, seed 42 |
| CV | Stratified 5-fold on train |
| Categorical missing | Explicit missing category |
| Numeric missing | Median, fitted in training |
| Encoding | One-hot |
| Numeric scaling | StandardScaler |
| Resampling | None |
| Test usage | Final benchmark only |

All seven models receive the **same raw cohort, split, and preprocessing contract**. This improves comparability, but the results remain a comparison of **prespecified baseline configurations**, not the maximum achievable performance of each algorithm family.


In [ ]:
preprocessing_summary = {
    "target_definition": "DIABETE4 1 vs 3",
    "n_primary_cohort": int(len(y)),
    "n_diabetes": int(y.sum()),
    "sample_diabetes_pct": round(float(y.mean() * 100), 4),
    "n_predictors_raw": len(PRIMARY_FEATURES),
    "complete_case_n": complete_case_n,
    "complete_case_pct": round(float(complete_case_pct), 4),
    "train_n": int(len(y_train)),
    "test_n": int(len(y_test)),
    "train_diabetes_pct": round(float(y_train.mean() * 100), 4),
    "test_diabetes_pct": round(float(y_test.mean() * 100), 4),
    "transformed_feature_count": int(len(feature_names)),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "cv_folds": 5,
    "categorical_missing_strategy": "constant -1 category",
    "numeric_missing_strategy": "median fit on training only",
    "categorical_encoding": "one-hot, handle_unknown=ignore",
    "numeric_scaling": "StandardScaler fit on training only",
}

with open("/kaggle/working/preprocessing_summary.json", "w", encoding="utf-8") as f:
    json.dump(preprocessing_summary, f, indent=2)

print(json.dumps(preprocessing_summary, indent=2))


# Part III — Seven Prespecified Model Baselines

## 3.0 Model roster

| # | Model | Family / role | Main precedent |
|---:|---|---|---|
| 01 | Logistic Regression | Linear probabilistic baseline | P01/P02/P03 |
| 02 | Decision Tree | Single non-linear tree | P02/P03 |
| 03 | Random Forest | Bagged tree ensemble | P02/P03 |
| 04 | Gradient Boosting | Sequential boosting | P03 + Friedman |
| 05 | XGBoost | Regularized boosted trees | P03/P05 + Chen & Guestrin |
| 06 | Linear SVM | Maximum-margin linear model | P02/P03 + Cortes & Vapnik |
| 07 | Gaussian Naive Bayes | Generative probabilistic baseline | P02 |

**This is not a reproduction of P03's exact seven-model roster.** P03 includes KNN; this study instead includes Gaussian NB from P02 to cover a distinct probabilistic/generative family and avoid a very costly instance-based baseline on 342k respondents.

## 3.1 Model 01 — Logistic Regression

**Why:** transparent linear/probabilistic baseline with direct BRFSS precedent.

$$
P(Y=1\mid X)=\sigma(\beta_0 + X\beta),\qquad
\sigma(z)=\frac{1}{1+e^{-z}}
$$

**Frozen configuration:** L2 penalty · `C=1` · `lbfgs` · no class weighting · no resampling · threshold 0.5.

**Provenance rule:** the model family is literature-grounded; the exact scikit-learn configuration is study-specific. P02 and P03 use different split/resampling/tuning protocols, so their published metrics are context—not numerical reproduction targets.


### 3.1.1 Evaluation Protocol

The model uses exactly the same frozen data protocol defined in Part II:

- training set: 274,031 respondents;
- internal test set: 68,508 respondents;
- preprocessing inside the model pipeline;
- Stratified 5-Fold CV on the training set;
- held-out test set evaluated after the baseline configuration is fixed;
- no resampling;
- probability threshold = 0.5.

Metrics:

- Accuracy
- Precision
- Recall / Sensitivity
- Specificity
- F1
- ROC-AUC
- Average Precision (AP)
- Brier score
- Confusion matrix

Cross-validation is summarized descriptively with fold values, mean, standard deviation, and range. Final-test uncertainty is estimated with a stratified bootstrap.


In [ ]:
from pathlib import Path
import platform
import sklearn
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    make_scorer,
)
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

MODEL_01_DIR = Path("/kaggle/working/model_01_logistic_regression")
MODEL_01_DIR.mkdir(parents=True, exist_ok=True)

logreg = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    class_weight=None,
)

logreg_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", logreg),
])

specificity_scorer = make_scorer(recall_score, pos_label=0)
precision_scorer = make_scorer(precision_score, zero_division=0)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": precision_scorer,
    "recall": "recall",
    "specificity": specificity_scorer,
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

environment = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "LogisticRegression",
    "solver": "lbfgs",
    "penalty": "l2",
    "C": 1.0,
    "max_iter": 1000,
    "class_weight": None,
    "threshold": 0.5,
    "random_state_split": RANDOM_STATE,
    "cv_folds": 5,
}

with open(MODEL_01_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2)

print(json.dumps(environment, indent=2))


CV_METRICS = [
    "accuracy", "balanced_accuracy", "precision", "recall",
    "specificity", "f1", "roc_auc", "pr_auc",
]
TEST_METRICS = CV_METRICS + ["brier_score"]

def summarize_cv(pipeline, X_data, y_data):
    """Run fold-isolated CV and return fold values plus descriptive mean/SD/range."""
    raw = cross_validate(
        pipeline,
        X_data,
        y_data,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=1,
    )
    folds = []
    for fold in range(cv.get_n_splits()):
        row = {"fold": fold + 1}
        for metric in CV_METRICS:
            row[metric] = float(raw[f"test_{metric}"][fold])
        row["fit_time_seconds"] = float(raw["fit_time"][fold])
        row["score_time_seconds"] = float(raw["score_time"][fold])
        folds.append(row)

    fold_df = pd.DataFrame(folds)
    summary = pd.DataFrame([
        {
            "metric": metric,
            "mean": float(fold_df[metric].mean()),
            "std": float(fold_df[metric].std(ddof=1)),
            "min": float(fold_df[metric].min()),
            "max": float(fold_df[metric].max()),
        }
        for metric in CV_METRICS
    ])
    return fold_df, summary

def point_metrics(y_true, pred, score, *, probability):
    """Common held-out metric contract for all seven models."""
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    out = {
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred)),
        "specificity": float(tn / (tn + fp)),
        "f1": float(f1_score(y_true, pred)),
        "roc_auc": float(roc_auc_score(y_true, score)),
        "pr_auc": float(average_precision_score(y_true, score)),
        "brier_score": float(brier_score_loss(y_true, score)) if probability else None,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    return out

def stratified_bootstrap(y_true, pred, score, point, *, probability, reps, seed):
    """Class-stratified respondent bootstrap; no model refitting or tuning."""
    y_np = np.asarray(y_true)
    pred = np.asarray(pred)
    score = np.asarray(score)
    pos = np.flatnonzero(y_np == 1)
    neg = np.flatnonzero(y_np == 0)
    rng = np.random.default_rng(seed)

    rows = []
    for b in range(reps):
        idx = np.concatenate([
            rng.choice(pos, size=len(pos), replace=True),
            rng.choice(neg, size=len(neg), replace=True),
        ])
        rng.shuffle(idx)
        m = point_metrics(
            y_np[idx],
            pred[idx],
            score[idx],
            probability=probability,
        )
        rows.append({"replicate": b + 1, **{k: m[k] for k in TEST_METRICS if m[k] is not None}})

    boot = pd.DataFrame(rows)
    summary_metrics = [m for m in TEST_METRICS if m in boot.columns]
    summary = pd.DataFrame([
        {
            "metric": metric,
            "point_estimate": point[metric],
            "bootstrap_mean": float(boot[metric].mean()),
            "ci95_lower": float(boot[metric].quantile(0.025)),
            "ci95_upper": float(boot[metric].quantile(0.975)),
        }
        for metric in summary_metrics
    ])
    return boot, summary


### 3.1.2 Five-Fold Cross-Validation on the Training Set

The entire model pipeline is passed into cross-validation. Therefore each fold independently fits:

1. categorical missing-value handling;
2. one-hot encoding;
3. numeric median imputation;
4. numeric scaling;
5. Logistic Regression.

The validation fold is never used to fit preprocessing statistics.


In [ ]:
logreg_cv_folds, logreg_cv_summary = summarize_cv(logreg_pipeline, X_train, y_train)
display(logreg_cv_folds)
display(logreg_cv_summary)
logreg_cv_folds.to_csv(MODEL_01_DIR / "cv_folds.csv", index=False)
logreg_cv_summary.to_csv(MODEL_01_DIR / "cv_summary.csv", index=False)


### 3.1.3 Fit the Fixed Baseline on the Full Training Set

After cross-validation, the same fixed configuration is fit once on the complete training partition. No CV or test metric is used to change its hyperparameters.

The held-out internal test set is used for the prespecified final evaluation at the standard probability threshold of 0.5; it is never used to tune the model or threshold.


In [ ]:
logreg_pipeline.fit(X_train, y_train)
y_test_prob = logreg_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= 0.5).astype(int)

test_point_metrics = point_metrics(
    y_test, y_test_pred, y_test_prob, probability=True
)
test_point_metrics["threshold"] = 0.5

print(json.dumps(test_point_metrics, indent=2))

with open(MODEL_01_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_point_metrics, f, indent=2)


### 3.1.4 Stratified Bootstrap Confidence Intervals

To quantify uncertainty on the held-out test set, we bootstrap diabetic and non-diabetic respondents separately with replacement. This preserves the test-set class composition while estimating sampling variability.

The test set is **not** used for parameter tuning; bootstrap samples are used only to estimate uncertainty around the already-fixed model's performance.


In [ ]:
BOOTSTRAP_REPS = 200
bootstrap_df, logreg_test_summary = stratified_bootstrap(
    y_test,
    y_test_pred,
    y_test_prob,
    test_point_metrics,
    probability=True,
    reps=BOOTSTRAP_REPS,
    seed=20251003,
)

display(logreg_test_summary)
bootstrap_df.to_csv(MODEL_01_DIR / "bootstrap_metrics.csv", index=False)
logreg_test_summary.to_csv(MODEL_01_DIR / "test_summary.csv", index=False)


### 3.1.5 Diagnostic Curves and Confusion Matrix

ROC and precision-recall curves are saved for comparison with later models. The confusion matrix uses the fixed 0.5 threshold.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob)

fig = plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC={test_point_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 01 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_01_DIR / "roc_curve.png", dpi=160)
plt.show()

fig = plt.figure(figsize=(6, 5))
plt.plot(recall_curve, precision_curve, label=f"Logistic Regression (AP={test_point_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 01 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_01_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[test_point_metrics["tn"], test_point_metrics["fp"]], [test_point_metrics["fn"], test_point_metrics["tp"]]])
fig = plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Model 01 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0, 1], ["No diabetes", "Diabetes"])
plt.yticks([0, 1], ["No diabetes", "Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_01_DIR / "confusion_matrix.png", dpi=160)
plt.show()


### 3.1.6 Model 01 Freeze Point

After this run, Model 01 is considered frozen as:

> **L2 Logistic Regression, C=1.0, lbfgs, no class weighting, no resampling, threshold=0.5**

The model is not retuned after seeing the internal test results.

Its role in the study is the transparent linear baseline against which non-linear and ensemble models will be compared.

The next model should be implemented only after this baseline's execution artifacts and results have been reviewed.


## 3.2 Model 02 — Decision Tree

**Role:** transparent non-linear baseline and explicit overfitting stress test.

| Item | Frozen choice |
|---|---|
| Criterion | Gini |
| Depth | Unrestricted |
| Min split / leaf | 2 / 1 |
| Pruning | None |
| Class weighting | None |
| Threshold | 0.5 |
| Seed | 42 |

P02/P03 support the **Decision Tree family**; this exact unpruned sklearn configuration is study-specific. Severe train–test divergence is expected to be audited rather than hidden.


### 3.2.1 Evaluation Protocol

Model 02 uses the identical frozen Part II cohort, split, features, and preprocessing used by Model 01.

This isolates the effect of the learning algorithm.

The primary comparison uses:

- Stratified 5-Fold CV on training data only;
- no resampling;
- fixed threshold 0.5;
- Accuracy, Precision, Recall, Specificity, F1, ROC-AUC, Average Precision (AP), and Brier score;
- untouched internal test evaluation;
- stratified bootstrap confidence intervals.

Because the tree is intentionally unpruned, we additionally record tree depth, leaf count, node count, and training performance as an overfitting diagnostic.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

MODEL_02_DIR = Path("/kaggle/working/model_02_decision_tree")
MODEL_02_DIR.mkdir(parents=True, exist_ok=True)

decision_tree = DecisionTreeClassifier(
    criterion="gini",
    splitter="best",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    class_weight=None,
    ccp_alpha=0.0,
    random_state=RANDOM_STATE,
)

decision_tree_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", decision_tree),
])

environment_model02 = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "DecisionTreeClassifier",
    "architecture": "CART-style binary classification tree",
    "criterion": "gini",
    "splitter": "best",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": None,
    "class_weight": None,
    "ccp_alpha": 0.0,
    "random_state": RANDOM_STATE,
    "threshold": 0.5,
    "cv_folds": 5,
}

with open(MODEL_02_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(environment_model02, f, indent=2)

print(json.dumps(environment_model02, indent=2))


### 3.2.2 Five-Fold Cross-Validation

As with Model 01, the complete preprocessing + Decision Tree pipeline is cross-validated.

Each fold independently learns:
- categorical missing handling;
- one-hot encoding;
- numeric imputation;
- scaling;
- tree splits.

No validation-fold or test-set information is used to fit the tree.


In [ ]:
dt_cv_folds, dt_cv_summary = summarize_cv(decision_tree_pipeline, X_train, y_train)
display(dt_cv_folds)
display(dt_cv_summary)
dt_cv_folds.to_csv(MODEL_02_DIR / "cv_folds.csv", index=False)
dt_cv_summary.to_csv(MODEL_02_DIR / "cv_summary.csv", index=False)


### 3.2.3 Fit the Frozen Tree on the Full Training Set

The Decision Tree is fit once on the complete training partition using the fixed configuration above.

We record tree complexity because unrestricted depth can produce a highly complex model and is a known reason single trees can overfit compared with ensemble methods.


In [ ]:
decision_tree_pipeline.fit(X_train, y_train)

fitted_tree = decision_tree_pipeline.named_steps["model"]

tree_complexity = {
    "max_depth": int(fitted_tree.get_depth()),
    "n_leaves": int(fitted_tree.get_n_leaves()),
    "node_count": int(fitted_tree.tree_.node_count),
}

train_prob_dt = decision_tree_pipeline.predict_proba(X_train)[:, 1]
train_pred_dt = (train_prob_dt >= 0.5).astype(int)

tree_complexity["train_accuracy"] = float(accuracy_score(y_train, train_pred_dt))
tree_complexity["train_roc_auc"] = float(roc_auc_score(y_train, train_prob_dt))

with open(MODEL_02_DIR / "tree_complexity.json", "w", encoding="utf-8") as f:
    json.dump(tree_complexity, f, indent=2)

print(json.dumps(tree_complexity, indent=2))


### 3.2.4 Held-Out Internal Test Evaluation

The internal test set is evaluated at the fixed probability threshold of 0.5.

No tree parameter, depth, pruning rule, or threshold is changed after viewing these results.


In [ ]:
y_test_prob_dt = decision_tree_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_dt = (y_test_prob_dt >= 0.5).astype(int)

dt_test_point_metrics = point_metrics(
    y_test, y_test_pred_dt, y_test_prob_dt, probability=True
)
dt_test_point_metrics["threshold"] = 0.5

print(json.dumps(dt_test_point_metrics, indent=2))
with open(MODEL_02_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(dt_test_point_metrics, f, indent=2)


### 3.2.5 Stratified Bootstrap Confidence Intervals

The same fixed-test bootstrap protocol used for Logistic Regression is reused for Decision Tree so uncertainty estimates are directly comparable.


In [ ]:
BOOTSTRAP_REPS = 200
dt_bootstrap_df, dt_test_summary = stratified_bootstrap(
    y_test,
    y_test_pred_dt,
    y_test_prob_dt,
    dt_test_point_metrics,
    probability=True,
    reps=BOOTSTRAP_REPS,
    seed=20251004,
)

display(dt_test_summary)
dt_bootstrap_df.to_csv(MODEL_02_DIR / "bootstrap_metrics.csv", index=False)
dt_test_summary.to_csv(MODEL_02_DIR / "test_summary.csv", index=False)


### 3.2.6 Diagnostic Curves, Confusion Matrix, and Tree Importance

ROC/PR curves and confusion matrix are saved for direct later comparison.

Impurity-based tree feature importance is exported only as a **model diagnostic**. It is not treated as causal evidence and is not used to change the frozen feature set.


In [ ]:
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_test_prob_dt)
precision_curve_dt, recall_curve_dt, _ = precision_recall_curve(y_test, y_test_prob_dt)

fig = plt.figure(figsize=(6, 5))
plt.plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC={dt_test_point_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 02 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_02_DIR / "roc_curve.png", dpi=160)
plt.show()

fig = plt.figure(figsize=(6, 5))
plt.plot(recall_curve_dt, precision_curve_dt, label=f"Decision Tree (AP={dt_test_point_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 02 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_02_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm_dt = np.array([[dt_test_point_metrics["tn"], dt_test_point_metrics["fp"]], [dt_test_point_metrics["fn"], dt_test_point_metrics["tp"]]])
fig = plt.figure(figsize=(5, 4))
plt.imshow(cm_dt)
plt.title("Model 02 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0, 1], ["No diabetes", "Diabetes"])
plt.yticks([0, 1], ["No diabetes", "Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_dt[i, j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_02_DIR / "confusion_matrix.png", dpi=160)
plt.show()

dt_feature_names = decision_tree_pipeline.named_steps["preprocess"].get_feature_names_out()
dt_importances = fitted_tree.feature_importances_

dt_importance_table = (
    pd.DataFrame({
        "transformed_feature": dt_feature_names,
        "gini_importance": dt_importances,
    })
    .sort_values("gini_importance", ascending=False)
)

display(dt_importance_table.head(20))
dt_importance_table.to_csv(MODEL_02_DIR / "feature_importance_transformed.csv", index=False)


### 3.2.7 Model 02 Freeze Point

After this run, Model 02 is frozen as:

> **Unpruned Gini CART-style Decision Tree, sklearn defaults made explicit, no class weighting, no resampling, threshold=0.5.**

Its scientific role is to test whether a single non-linear hierarchical partitioning model improves on the linear Logistic Regression baseline, while also exposing the instability/overfitting risk that motivates later ensemble tree methods.

No post-test pruning or depth tuning is allowed for Model 02. Any tuned/pruned tree would be a separate experiment.


## 3.3 Model 03 — Random Forest

**Role:** bagging-based extension of the single-tree baseline.

| Item | Frozen choice |
|---|---|
| Trees | 100 |
| Criterion | Gini |
| Feature sampling | `sqrt` |
| Bootstrap | Yes |
| Tree depth | Unrestricted |
| Class weighting | None |
| Threshold | 0.5 |
| Seed | 42 |

P02/P03 support Random Forest as a BRFSS comparator; the exact configuration is a fixed study baseline, not a tuned reproduction.


### 3.3.1 Evaluation Protocol

To preserve a fair within-study comparison, Random Forest uses exactly the same:

- BRFSS 2025 target cohort;
- frozen 20 predictors;
- 80/20 stratified split;
- Part II preprocessing;
- Stratified 5-Fold CV;
- natural class distribution;
- evaluation metrics;
- held-out test set.

The preprocessing pipeline remains identical even though tree models do not require standardization. This keeps the benchmark input representation fixed across model families.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

MODEL_03_DIR = Path("/kaggle/working/model_03_random_forest")
MODEL_03_DIR.mkdir(parents=True, exist_ok=True)

random_forest = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    class_weight=None,
    ccp_alpha=0.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

random_forest_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", random_forest),
])

environment_model03 = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "RandomForestClassifier",
    "architecture": "bootstrap-aggregated randomized decision-tree ensemble",
    "n_estimators": 100,
    "criterion": "gini",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "bootstrap": True,
    "class_weight": None,
    "ccp_alpha": 0.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "threshold": 0.5,
    "cv_folds": 5,
}

with open(MODEL_03_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(environment_model03, f, indent=2)

print(json.dumps(environment_model03, indent=2))


### 3.3.2 Five-Fold Cross-Validation

The full preprocessing + Random Forest pipeline is evaluated under the same five stratified folds used for the previous models.

Each fold learns preprocessing and all 100 trees only from that fold's fitting partition.


In [ ]:
rf_cv_folds, rf_cv_summary = summarize_cv(random_forest_pipeline, X_train, y_train)
display(rf_cv_folds)
display(rf_cv_summary)
rf_cv_folds.to_csv(MODEL_03_DIR / "cv_folds.csv", index=False)
rf_cv_summary.to_csv(MODEL_03_DIR / "cv_summary.csv", index=False)


### 3.3.3 Fit the Frozen Forest and Inspect Ensemble Complexity

After CV, the same fixed forest is fit once on the complete training partition.

We record aggregate component-tree complexity to compare with the extreme overfitting observed in Model 02. Individual Random Forest trees can still be deep; the key architectural difference is averaging many decorrelated trees rather than relying on one tree.


In [ ]:
random_forest_pipeline.fit(X_train, y_train)

fitted_forest = random_forest_pipeline.named_steps["model"]

tree_depths = np.array([est.get_depth() for est in fitted_forest.estimators_])
tree_leaves = np.array([est.get_n_leaves() for est in fitted_forest.estimators_])
tree_nodes = np.array([est.tree_.node_count for est in fitted_forest.estimators_])

train_prob_rf = random_forest_pipeline.predict_proba(X_train)[:, 1]
train_pred_rf = (train_prob_rf >= 0.5).astype(int)

forest_complexity = {
    "n_estimators": int(len(fitted_forest.estimators_)),
    "mean_tree_depth": float(tree_depths.mean()),
    "min_tree_depth": int(tree_depths.min()),
    "max_tree_depth": int(tree_depths.max()),
    "mean_leaves_per_tree": float(tree_leaves.mean()),
    "mean_nodes_per_tree": float(tree_nodes.mean()),
    "train_accuracy": float(accuracy_score(y_train, train_pred_rf)),
    "train_roc_auc": float(roc_auc_score(y_train, train_prob_rf)),
}

with open(MODEL_03_DIR / "forest_complexity.json", "w", encoding="utf-8") as f:
    json.dump(forest_complexity, f, indent=2)

print(json.dumps(forest_complexity, indent=2))


### 3.3.4 Held-Out Internal Test Evaluation

The held-out test set is evaluated at the fixed 0.5 threshold after model specification; it is not used for tuning.

No forest parameter or threshold is changed after test metrics are observed.


In [ ]:
y_test_prob_rf = random_forest_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_rf = (y_test_prob_rf >= 0.5).astype(int)

rf_test_point_metrics = point_metrics(
    y_test, y_test_pred_rf, y_test_prob_rf, probability=True
)
rf_test_point_metrics["threshold"] = 0.5

print(json.dumps(rf_test_point_metrics, indent=2))
with open(MODEL_03_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(rf_test_point_metrics, f, indent=2)


### 3.3.5 Stratified Bootstrap Confidence Intervals

The same 200-replicate stratified bootstrap protocol is applied to the fixed Random Forest test predictions.


In [ ]:
BOOTSTRAP_REPS = 200
rf_bootstrap_df, rf_test_summary = stratified_bootstrap(
    y_test,
    y_test_pred_rf,
    y_test_prob_rf,
    rf_test_point_metrics,
    probability=True,
    reps=BOOTSTRAP_REPS,
    seed=20251005,
)

display(rf_test_summary)
rf_bootstrap_df.to_csv(MODEL_03_DIR / "bootstrap_metrics.csv", index=False)
rf_test_summary.to_csv(MODEL_03_DIR / "test_summary.csv", index=False)


### 3.3.6 Diagnostic Curves, Confusion Matrix, and Forest Importance

ROC/PR curves and the confusion matrix are saved for later model-comparison figures.

Impurity-based Random Forest feature importance is exported as a diagnostic only. It is not causal evidence and is not used to modify the frozen predictor set.


In [ ]:
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_test_prob_rf)
precision_curve_rf, recall_curve_rf, _ = precision_recall_curve(y_test, y_test_prob_rf)

fig = plt.figure(figsize=(6, 5))
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={rf_test_point_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 03 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_03_DIR / "roc_curve.png", dpi=160)
plt.show()

fig = plt.figure(figsize=(6, 5))
plt.plot(recall_curve_rf, precision_curve_rf, label=f"Random Forest (AP={rf_test_point_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 03 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_03_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm_rf = np.array([[rf_test_point_metrics["tn"], rf_test_point_metrics["fp"]], [rf_test_point_metrics["fn"], rf_test_point_metrics["tp"]]])
fig = plt.figure(figsize=(5, 4))
plt.imshow(cm_rf)
plt.title("Model 03 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0, 1], ["No diabetes", "Diabetes"])
plt.yticks([0, 1], ["No diabetes", "Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_rf[i, j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_03_DIR / "confusion_matrix.png", dpi=160)
plt.show()

rf_feature_names = random_forest_pipeline.named_steps["preprocess"].get_feature_names_out()
rf_importances = fitted_forest.feature_importances_

rf_importance_table = (
    pd.DataFrame({
        "transformed_feature": rf_feature_names,
        "gini_importance": rf_importances,
    })
    .sort_values("gini_importance", ascending=False)
)

display(rf_importance_table.head(20))
rf_importance_table.to_csv(MODEL_03_DIR / "feature_importance_transformed.csv", index=False)


### 3.3.7 Model 03 Freeze Point

After this verified run, Model 03 is frozen as:

> **100-tree bootstrap Random Forest, Gini splits, sqrt feature sampling, unrestricted component-tree depth, no class weighting, no resampling, threshold=0.5.**

Its role is to test whether ensemble averaging substantially improves generalization over the overfit single Decision Tree baseline.

No post-test change to tree count, depth, feature sampling, class weights, or threshold is permitted. Any tuned Random Forest must be recorded as a separate experiment.


## 3.4–3.7 Remaining Frozen Model Families

Models 04–07 continue **inside this same notebook**. They reuse the Part II cohort, train/test partition, preprocessing transformer definition, CV folds, scoring rules, and held-out test set already established above.

The model-specific sections below therefore define only the estimator architecture and its evaluation. They do **not** reload BRFSS, rebuild the cohort, or create a different split.


In [ ]:
# Shared imports/helpers for Models 04–07 and the paired comparison.
from itertools import combinations
from scipy.stats import binomtest
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
import xgboost as xgb

def to_dense(X_matrix):
    return X_matrix.toarray() if hasattr(X_matrix, "toarray") else np.asarray(X_matrix)

dense_transformer = FunctionTransformer(
    to_dense,
    accept_sparse=True,
    feature_names_out="one-to-one",
)

print("Additional frozen model families ready.")


## 3.4 Model 04 — Gradient Boosting

**Role:** classical sequential boosting baseline.

- BRFSS precedent: P03.
- Architecture reference: Friedman (2001), DOI `10.1214/aos/1013203451`.
- Frozen: 100 stages, learning rate 0.1, depth-3 weak learners, no subsampling, threshold 0.5.
- Exact parameters are study-specific and are **not** claimed to reproduce P03 tuning.


### 3.4.1 Frozen Study Protocol

The following are inherited unchanged from Part II:

- target: `DIABETE4=1` versus `DIABETE4=3`;
- 20 frozen predictors;
- 80/20 stratified split with `random_state=42`;
- categorical missing values represented explicitly and one-hot encoded;
- continuous missing values median-imputed;
- continuous predictors standardized;
- Stratified 5-Fold CV on training data only;
- no SMOTE or class resampling in the primary benchmark;
- held-out test set evaluated once after model specification is frozen.

The test set is never used to select Gradient Boosting hyperparameters.


### 3.4.2 Model Architecture

The fixed baseline is:

```python
GradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.1,
    n_estimators=100,
    subsample=1.0,
    criterion="friedman_mse",
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=3,
    max_features=None,
    random_state=42,
)
```

### Provenance classification

**Literature/architecture-supported**
- Gradient Boosting model family;
- sequential additive trees minimizing classification loss;
- use on BRFSS diabetes;
- 5-fold CV and held-out evaluation.

**Study-specific fixed choices**
- 100 stages;
- learning rate 0.1;
- depth-3 weak learners;
- no stochastic subsampling;
- no class weighting/resampling;
- threshold 0.5.

These values are fixed before test evaluation and are not claimed to reproduce the tuned P03 configuration.


In [ ]:
MODEL_04_DIR = Path("/kaggle/working/model_04_gradient_boosting")
MODEL_04_DIR.mkdir(parents=True, exist_ok=True)

gradient_boosting = GradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.1,
    n_estimators=100,
    subsample=1.0,
    criterion="friedman_mse",
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=3,
    max_features=None,
    random_state=RANDOM_STATE,
)

gb_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", gradient_boosting),
])

gb_config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "GradientBoostingClassifier",
    "architecture": "forward stage-wise additive regression-tree ensemble",
    "loss": "log_loss",
    "learning_rate": 0.1,
    "n_estimators": 100,
    "subsample": 1.0,
    "criterion": "friedman_mse",
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_depth": 3,
    "max_features": None,
    "random_state": RANDOM_STATE,
    "threshold": 0.5,
    "resampling": "none",
    "cv_folds": 5,
}
with open(MODEL_04_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(gb_config, f, indent=2)

print(json.dumps(gb_config, indent=2))


### 3.4.3 Five-Fold Cross-Validation

The full preprocessing + Gradient Boosting pipeline is fitted independently inside each training fold.

No preprocessing statistic or tree is learned from the validation fold or final test set.


In [ ]:
gb_cv_folds, gb_cv_summary = summarize_cv(gb_pipeline, X_train, y_train)
display(gb_cv_folds)
display(gb_cv_summary)
gb_cv_folds.to_csv(MODEL_04_DIR / "cv_folds.csv", index=False)
gb_cv_summary.to_csv(MODEL_04_DIR / "cv_summary.csv", index=False)


### 3.4.4 Fit Frozen Model and Evaluate the Held-Out Test Set

After CV, the exact same fixed configuration is fitted once on all training respondents.

The held-out test set is evaluated at probability threshold 0.5 after specification is fixed. No hyperparameter or threshold is changed using these results.


In [ ]:
gb_pipeline.fit(X_train, y_train)
y_test_prob_gb = gb_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_gb = (y_test_prob_gb >= 0.5).astype(int)

gb_test_metrics = point_metrics(
    y_test, y_test_pred_gb, y_test_prob_gb, probability=True
)
gb_test_metrics["threshold"] = 0.5

print(json.dumps(gb_test_metrics, indent=2))
with open(MODEL_04_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(gb_test_metrics, f, indent=2)


### 3.4.5 Stratified Bootstrap Confidence Intervals

As with Models 01–03, uncertainty on fixed test predictions is estimated with 200 stratified bootstrap replicates. This procedure estimates uncertainty only; it does not tune the model.


In [ ]:
BOOTSTRAP_REPS = 200
gb_bootstrap_df, gb_test_summary = stratified_bootstrap(
    y_test,
    y_test_pred_gb,
    y_test_prob_gb,
    gb_test_metrics,
    probability=True,
    reps=BOOTSTRAP_REPS,
    seed=20251006,
)

display(gb_test_summary)
gb_bootstrap_df.to_csv(MODEL_04_DIR / "bootstrap_metrics.csv", index=False)
gb_test_summary.to_csv(MODEL_04_DIR / "test_summary.csv", index=False)


### 3.4.6 Diagnostics

ROC/PR curves and the fixed-threshold confusion matrix are saved.

Gradient Boosting impurity importance is exported as a diagnostic only. It is not causal evidence and is not used to reselect predictors.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob_gb)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob_gb)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Gradient Boosting (AUC={gb_test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 04 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_04_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"Gradient Boosting (AP={gb_test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 04 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_04_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[gb_test_metrics["tn"], gb_test_metrics["fp"]], [gb_test_metrics["fn"], gb_test_metrics["tp"]]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 04 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_04_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = gb_pipeline.named_steps["preprocess"].get_feature_names_out()
importances = gb_pipeline.named_steps["model"].feature_importances_
importance_table = pd.DataFrame({
    "transformed_feature": feature_names,
    "gini_importance": importances,
}).sort_values("gini_importance", ascending=False)

display(importance_table.head(20))
importance_table.to_csv(MODEL_04_DIR / "feature_importance_transformed.csv", index=False)


### 3.4.7 Freeze Rule

After this notebook completes successfully, Model 04 is frozen as:

> **100-stage Gradient Boosting classifier, log-loss objective, learning rate 0.1, depth-3 regression-tree weak learners, no resampling, threshold 0.5.**

No post-test change to stage count, learning rate, tree depth, threshold, preprocessing, or feature set is allowed. Any tuned Gradient Boosting configuration must be versioned as a separate experiment.


## 3.5 Model 05 — XGBoost

**Role:** regularized gradient-boosted-tree baseline.

- BRFSS precedents: P03 and P05.
- Architecture: Chen & Guestrin (2016), DOI `10.1145/2939672.2939785`.
- Frozen: 100 trees, learning rate 0.3, max depth 6, `hist` tree method, default-style L2 regularization, threshold 0.5.
- This is plain XGBoost—**not** GA-XGBoost and not a tuned reproduction of P03/P05.


### 3.5.1 Frozen Study Protocol

Inherited unchanged from Part II:

- target: `DIABETE4=1` vs `DIABETE4=3`;
- 20 frozen predictors;
- 80/20 stratified split, `random_state=42`;
- categorical missing category + one-hot encoding;
- continuous median imputation + scaling;
- Stratified 5-Fold CV on training data only;
- natural class distribution;
- no SMOTE/undersampling;
- held-out test set used only after specification is frozen.

This allows a direct within-study comparison with Models 01–04.


### 3.5.2 Model Architecture

XGBoost extends gradient tree boosting with a regularized objective and systems optimizations for scalable tree construction. The original Chen & Guestrin formulation includes regularization on tree complexity and second-order optimization of the boosting objective.

The fixed baseline used here is:

```python
XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    booster="gbtree",
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    min_child_weight=1,
    gamma=0.0,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)
```

### Provenance classification

**Literature-supported**
- XGBoost model family and regularized boosting architecture;
- direct BRFSS diabetes use in P03 and P05;
- stratified CV / held-out evaluation.

**Study-specific fixed choices**
- 100 trees;
- learning rate 0.3;
- max depth 6;
- default-style L2 regularization;
- histogram tree builder for efficient execution;
- no SMOTE;
- threshold 0.5.

This is a plain XGBoost baseline, not GA-XGBoost and not a tuned reproduction.


In [ ]:
MODEL_05_DIR = Path("/kaggle/working/model_05_xgboost")
MODEL_05_DIR.mkdir(parents=True, exist_ok=True)

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    booster="gbtree",
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    min_child_weight=1,
    gamma=0.0,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)

xgb_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", xgb_model),
])

xgb_config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "model": "XGBClassifier",
    "architecture": "regularized gradient-boosted decision-tree ensemble",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "booster": "gbtree",
    "n_estimators": 100,
    "learning_rate": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "gamma": 0.0,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "threshold": 0.5,
    "resampling": "none",
    "cv_folds": 5,
}
with open(MODEL_05_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(xgb_config, f, indent=2)

print(json.dumps(xgb_config, indent=2))


### 3.5.3 Five-Fold Cross-Validation

The complete preprocessing + XGBoost pipeline is re-fit independently inside each training fold.

No validation-fold or test-set information is used to select trees or preprocessing statistics.


In [ ]:
xgb_cv_folds, xgb_cv_summary = summarize_cv(xgb_pipeline, X_train, y_train)
display(xgb_cv_folds)
display(xgb_cv_summary)
xgb_cv_folds.to_csv(MODEL_05_DIR / "cv_folds.csv", index=False)
xgb_cv_summary.to_csv(MODEL_05_DIR / "cv_summary.csv", index=False)


### 3.5.4 Fit Frozen Model and Evaluate the Held-Out Test Set

After CV, the exact same fixed configuration is fit once on the full training set.

The held-out test set is evaluated at threshold 0.5 after specification is fixed. No post-test parameter adjustment is permitted.


In [ ]:
xgb_pipeline.fit(X_train, y_train)
y_test_prob_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_xgb = (y_test_prob_xgb >= 0.5).astype(int)

xgb_test_metrics = point_metrics(
    y_test, y_test_pred_xgb, y_test_prob_xgb, probability=True
)
xgb_test_metrics["threshold"] = 0.5

print(json.dumps(xgb_test_metrics, indent=2))
with open(MODEL_05_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(xgb_test_metrics, f, indent=2)


### 3.5.5 Stratified Bootstrap Confidence Intervals

Two hundred class-stratified bootstrap samples estimate uncertainty around the already-fixed held-out predictions. They are not used for tuning.


In [ ]:
BOOTSTRAP_REPS = 200
xgb_bootstrap_df, xgb_test_summary = stratified_bootstrap(
    y_test,
    y_test_pred_xgb,
    y_test_prob_xgb,
    xgb_test_metrics,
    probability=True,
    reps=BOOTSTRAP_REPS,
    seed=20251007,
)

display(xgb_test_summary)
xgb_bootstrap_df.to_csv(MODEL_05_DIR / "bootstrap_metrics.csv", index=False)
xgb_test_summary.to_csv(MODEL_05_DIR / "test_summary.csv", index=False)


### 3.5.6 Diagnostics

The ROC curve, precision-recall curve, confusion matrix, and transformed-feature importance are exported.

XGBoost feature importance is a model diagnostic only and is not causal evidence.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob_xgb)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob_xgb)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"XGBoost (AUC={xgb_test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 05 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_05_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"XGBoost (AP={xgb_test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 05 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_05_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[xgb_test_metrics["tn"], xgb_test_metrics["fp"]], [xgb_test_metrics["fn"], xgb_test_metrics["tp"]]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 05 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_05_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = xgb_pipeline.named_steps["preprocess"].get_feature_names_out()
importances = xgb_pipeline.named_steps["model"].feature_importances_
importance_table = pd.DataFrame({
    "transformed_feature": feature_names,
    "xgboost_importance": importances,
}).sort_values("xgboost_importance", ascending=False)

display(importance_table.head(20))
importance_table.to_csv(MODEL_05_DIR / "feature_importance_transformed.csv", index=False)


### 3.5.7 Freeze Rule

After verified execution, Model 05 is frozen as:

> **100-tree XGBoost gbtree classifier, binary logistic objective, learning rate 0.3, max depth 6, L2 regularization 1.0, histogram tree builder, no resampling, threshold 0.5.**

No post-test change to depth, learning rate, regularization, sampling, threshold, preprocessing, or feature set is allowed.

Any GA-XGBoost, tuned XGBoost, class-weighted XGBoost, or resampled XGBoost must be recorded as a separate experiment.


## 3.6 Model 06 — Linear Support Vector Machine

**Role:** large-sample sparse maximum-margin baseline.

- P02 directly compared linear/RBF/polynomial SVM; linear had strong BRFSS precedent.
- P03 supports SVM as a comparator.
- Architecture reference: Cortes & Vapnik (1995), DOI `10.1007/BF00994018`.
- Frozen: L2 squared-hinge, `C=1`, no class weighting, decision boundary = 0.

**Metric rule:** use `decision_function` for ROC-AUC/Average Precision (AP). Do **not** report Brier score because the margin is not a calibrated probability.


### 3.6.1 Frozen Study Protocol

Inherited unchanged from Part II:

- `DIABETE4=1` vs `DIABETE4=3`;
- frozen 20 predictors;
- 80/20 stratified split with `random_state=42`;
- categorical missing category + one-hot encoding;
- continuous median imputation + standardization;
- Stratified 5-Fold CV;
- no SMOTE or class reweighting;
- held-out internal test set.

### Important metric difference

`LinearSVC` does not produce calibrated probabilities.

Therefore:
- Accuracy, Precision, Recall, Specificity, and F1 use predicted labels.
- ROC-AUC and Average Precision (AP) use the continuous `decision_function` margin.
- The class boundary is **decision score = 0**.
- **Brier score is not reported** because applying it to uncalibrated margins would be invalid.

Probability calibration, if desired, must be a separate experiment.


### 3.6.2 Model Architecture

Support Vector Machines seek a separating hyperplane with a large margin between classes, while soft-margin regularization allows classification violations for non-separable data.

The fixed large-sample linear baseline is:

```python
LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual="auto",
    tol=1e-4,
    C=1.0,
    fit_intercept=True,
    class_weight=None,
    random_state=42,
    max_iter=10000,
)
```

### Provenance classification

**Literature-supported**
- SVM family and maximum-margin architecture;
- direct linear SVM use on BRFSS diabetes in P02;
- SVM comparator role in P03.

**Study-specific fixed choices**
- `LinearSVC` implementation;
- L2 + squared-hinge objective;
- `C=1.0`;
- automatic primal/dual selection;
- no class weighting;
- no SMOTE;
- zero-margin class threshold.

No RBF/poly kernel or calibration is added to this primary baseline.


In [ ]:
MODEL_06_DIR = Path("/kaggle/working/model_06_linear_svm")
MODEL_06_DIR.mkdir(parents=True, exist_ok=True)

linear_svm = LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual="auto",
    tol=1e-4,
    C=1.0,
    fit_intercept=True,
    class_weight=None,
    random_state=RANDOM_STATE,
    max_iter=10000,
)

svm_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", linear_svm),
])

svm_config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "LinearSVC",
    "architecture": "soft-margin linear support vector classifier",
    "penalty": "l2",
    "loss": "squared_hinge",
    "dual": "auto",
    "tol": 1e-4,
    "C": 1.0,
    "fit_intercept": True,
    "class_weight": None,
    "random_state": RANDOM_STATE,
    "max_iter": 10000,
    "decision_threshold": 0.0,
    "probability_calibration": None,
    "resampling": "none",
    "cv_folds": 5,
}
with open(MODEL_06_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(svm_config, f, indent=2)

print(json.dumps(svm_config, indent=2))


### 3.6.3 Five-Fold Cross-Validation

The full preprocessing + Linear SVM pipeline is fitted independently inside every training fold.

ROC-AUC and Average Precision (AP) are computed from the signed decision margin rather than probabilities.


In [ ]:
svm_cv_folds, svm_cv_summary = summarize_cv(svm_pipeline, X_train, y_train)
display(svm_cv_folds)
display(svm_cv_summary)
svm_cv_folds.to_csv(MODEL_06_DIR / "cv_folds.csv", index=False)
svm_cv_summary.to_csv(MODEL_06_DIR / "cv_summary.csv", index=False)


### 3.6.4 Fit Frozen Model and Evaluate the Held-Out Test Set

The fixed model is fit once on the full training partition.

Class labels use the natural SVM boundary at decision margin zero. Ranking metrics use the continuous decision margin.


In [ ]:
svm_pipeline.fit(X_train, y_train)
y_test_score_svm = svm_pipeline.decision_function(X_test)
y_test_pred_svm = svm_pipeline.predict(X_test)

svm_test_metrics = point_metrics(
    y_test, y_test_pred_svm, y_test_score_svm, probability=False
)
svm_test_metrics["decision_threshold"] = 0.0
svm_test_metrics["n_iter"] = int(
    np.max(np.atleast_1d(svm_pipeline.named_steps["model"].n_iter_))
)

print(json.dumps(svm_test_metrics, indent=2))
with open(MODEL_06_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(svm_test_metrics, f, indent=2)


### 3.6.5 Stratified Bootstrap Confidence Intervals

The same 200-replicate class-stratified bootstrap is used for all valid Linear SVM metrics.

Brier score is omitted because the SVM margins are not calibrated probabilities.


In [ ]:
BOOTSTRAP_REPS = 200
svm_bootstrap_df, svm_test_summary = stratified_bootstrap(
    y_test,
    y_test_pred_svm,
    y_test_score_svm,
    svm_test_metrics,
    probability=False,
    reps=BOOTSTRAP_REPS,
    seed=20251008,
)

display(svm_test_summary)
svm_bootstrap_df.to_csv(MODEL_06_DIR / "bootstrap_metrics.csv", index=False)
svm_test_summary.to_csv(MODEL_06_DIR / "test_summary.csv", index=False)


### 3.6.6 Diagnostics

ROC and precision-recall curves use the SVM decision score.

The transformed-feature coefficients describe the fitted hyperplane. Their magnitudes are model diagnostics, not causal effects.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_score_svm)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_score_svm)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Linear SVM (AUC={svm_test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 06 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_06_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"Linear SVM (AP={svm_test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 06 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_06_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[svm_test_metrics["tn"], svm_test_metrics["fp"]], [svm_test_metrics["fn"], svm_test_metrics["tp"]]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 06 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_06_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = svm_pipeline.named_steps["preprocess"].get_feature_names_out()
coef = svm_pipeline.named_steps["model"].coef_.ravel()
coef_table = pd.DataFrame({
    "transformed_feature": feature_names,
    "coefficient": coef,
    "abs_coefficient": np.abs(coef),
}).sort_values("abs_coefficient", ascending=False)

display(coef_table.head(20))
coef_table.to_csv(MODEL_06_DIR / "linear_coefficients.csv", index=False)


### 3.6.7 Freeze Rule

After verified execution, Model 06 is frozen as:

> **L2 Linear SVM with squared-hinge loss, C=1.0, automatic primal/dual selection, no class weighting, no resampling, zero-margin decision boundary.**

No post-test change to `C`, loss, class weighting, calibration, decision threshold, preprocessing, or feature set is allowed.

RBF/poly SVMs or probability-calibrated SVMs must be separate experiments.


## 3.7 Model 07 — Gaussian Naive Bayes

**Role:** simple generative/probabilistic baseline with a deliberately different inductive bias.

- Direct BRFSS precedent: P02.
- Frozen: empirical priors, `var_smoothing=1e-9`, threshold 0.5.
- Same frozen preprocessing; sparse output is converted to dense only for estimator compatibility.

**Assumption caveat:** many one-hot features are binary, not Gaussian. GNB is retained as a baseline—not as a claim that all transformed BRFSS features satisfy Gaussian class-conditional assumptions.


### 3.7.1 Frozen Study Protocol

Inherited unchanged from Part II:

- target: `DIABETE4=1` vs `DIABETE4=3`;
- same 20 predictors;
- 80/20 stratified split with `random_state=42`;
- same missing-value handling;
- same one-hot categorical representation;
- same numeric scaling;
- Stratified 5-Fold CV on training data;
- no SMOTE or class balancing;
- held-out test set.

### Dense conversion

The frozen preprocessing pipeline outputs a sparse matrix because of one-hot encoding.

`GaussianNB` requires dense input, so Model 07 adds only:

```text
frozen Part II preprocessing
        ↓
sparse matrix → dense array
        ↓
GaussianNB
```

No values, categories, scaling rules, or predictors are changed.


### 3.7.2 Model Architecture

Naive Bayes applies Bayes' rule under the simplifying assumption that predictors are conditionally independent given the outcome class.

Gaussian Naive Bayes additionally models each feature's class-conditional distribution as Gaussian.

The fixed baseline is:

```python
GaussianNB(
    priors=None,
    var_smoothing=1e-9,
)
```

### Provenance classification

**Literature-supported**
- Gaussian Naive Bayes directly used on BRFSS diabetes in P02;
- probabilistic Naive Bayes classifier family;
- held-out evaluation.

**Study-specific**
- sklearn `GaussianNB`;
- default empirical class priors;
- `var_smoothing=1e-9`;
- sparse-to-dense adapter;
- no SMOTE;
- probability threshold 0.5.

### Assumption caveat

After one-hot encoding, many transformed features are binary rather than truly Gaussian. Model 07 is therefore treated as a simple probabilistic baseline, not as a claim that the Gaussian feature-distribution assumption is scientifically correct for every BRFSS indicator.


In [ ]:
MODEL_07_DIR = Path("/kaggle/working/model_07_gaussian_naive_bayes")
MODEL_07_DIR.mkdir(parents=True, exist_ok=True)

gnb = GaussianNB(
    priors=None,
    var_smoothing=1e-9,
)

gnb_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("to_dense", dense_transformer),
    ("model", gnb),
])

gnb_config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "GaussianNB",
    "architecture": "Gaussian class-conditional naive Bayes classifier",
    "priors": None,
    "var_smoothing": 1e-9,
    "dense_adapter": True,
    "threshold": 0.5,
    "resampling": "none",
    "cv_folds": 5,
}

with open(MODEL_07_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(gnb_config, f, indent=2)

print(json.dumps(gnb_config, indent=2))


### 3.7.3 Five-Fold Cross-Validation

The complete preprocessing + dense conversion + GaussianNB pipeline is re-fit independently inside each training fold.

Dense conversion is representation-only: it does not alter feature values.


In [ ]:
gnb_cv_folds, gnb_cv_summary = summarize_cv(gnb_pipeline, X_train, y_train)
display(gnb_cv_folds)
display(gnb_cv_summary)
gnb_cv_folds.to_csv(MODEL_07_DIR / "cv_folds.csv", index=False)
gnb_cv_summary.to_csv(MODEL_07_DIR / "cv_summary.csv", index=False)


### 3.7.4 Fit Frozen Model and Evaluate the Held-Out Test Set

The exact same fixed model is fit once on all training respondents.

The test set is evaluated once at probability threshold 0.5. No prior, smoothing, or threshold value is changed afterward.


In [ ]:
gnb_pipeline.fit(X_train, y_train)
y_test_prob_gnb = gnb_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_gnb = (y_test_prob_gnb >= 0.5).astype(int)

gnb_test_metrics = point_metrics(
    y_test, y_test_pred_gnb, y_test_prob_gnb, probability=True
)
gnb_test_metrics["threshold"] = 0.5

print(json.dumps(gnb_test_metrics, indent=2))
with open(MODEL_07_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(gnb_test_metrics, f, indent=2)


### 3.7.5 Stratified Bootstrap Confidence Intervals

Two hundred class-stratified bootstrap samples quantify uncertainty around the fixed held-out predictions.


In [ ]:
BOOTSTRAP_REPS = 200
gnb_bootstrap_df, gnb_test_summary = stratified_bootstrap(
    y_test,
    y_test_pred_gnb,
    y_test_prob_gnb,
    gnb_test_metrics,
    probability=True,
    reps=BOOTSTRAP_REPS,
    seed=20251009,
)

display(gnb_test_summary)
gnb_bootstrap_df.to_csv(MODEL_07_DIR / "bootstrap_metrics.csv", index=False)
gnb_test_summary.to_csv(MODEL_07_DIR / "test_summary.csv", index=False)


### 3.7.6 Diagnostics

ROC, precision-recall, confusion matrix, and class-conditional Gaussian parameters are saved.

The learned means/variances are descriptive model parameters only and are not causal effects.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob_gnb)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob_gnb)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Gaussian NB (AUC={gnb_test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 07 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_07_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"Gaussian NB (AP={gnb_test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 07 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_07_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[gnb_test_metrics["tn"], gnb_test_metrics["fp"]], [gnb_test_metrics["fn"], gnb_test_metrics["tp"]]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 07 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_07_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = gnb_pipeline.named_steps["preprocess"].get_feature_names_out()
model = gnb_pipeline.named_steps["model"]
params = pd.DataFrame({
    "transformed_feature": feature_names,
    "mean_no_diabetes": model.theta_[0],
    "mean_diabetes": model.theta_[1],
    "variance_no_diabetes": model.var_[0],
    "variance_diabetes": model.var_[1],
})
params.to_csv(MODEL_07_DIR / "gaussian_parameters.csv", index=False)
display(params.head(20))


### 3.7.7 Freeze Rule

After verified execution, Model 07 is frozen as:

> **Gaussian Naive Bayes with empirical class priors, var_smoothing=1e-9, dense adapter after frozen preprocessing, no resampling, threshold=0.5.**

No post-test changes are allowed to priors, smoothing, threshold, preprocessing, or feature set.

This completes the **seven-model primary benchmark**. Any KNN, neural network, calibrated model, tuned model, or resampling variant belongs to sensitivity/secondary experiments rather than the primary comparison.


## 3.8 Model-development audit checkpoint

Before model-to-model inference:

- all seven estimators use the same cohort/split/preprocessing contract;
- no model uses SMOTE, class weights, or test-set tuning;
- Linear SVM uses margins, not pseudo-probabilities;
- Gaussian NB's distributional mismatch is explicitly labeled;
- Decision Tree / Random Forest train–test gaps are retained as overfitting diagnostics;
- frozen point estimates are reconciled before paired comparison;
- published paper metrics are **context only**, because study protocols are not equivalent.


# Part IV — Consolidated Seven-Model Comparison and Paired Statistical Analysis

The seven primary models are now frozen. This section **does not fit them again**.

Instead, it reuses the respondent-level held-out predictions/scores already generated in Sections 3.1–3.7. This has three advantages:

- every comparison uses the exact same 68,508 held-out respondents;
- no second implementation of a model can silently drift from its frozen configuration;
- the comparison directly benefits from work already performed in the model sections.

Before any paired inference, the notebook reconciles the current point metrics with the previously verified frozen benchmark. Any material drift stops execution.


In [ ]:
COMPARISON_DIR = Path("/kaggle/working/model_comparison")
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = [
    "logistic_regression",
    "decision_tree",
    "random_forest",
    "gradient_boosting",
    "xgboost",
    "linear_svm",
    "gaussian_naive_bayes",
]
DISPLAY_NAMES = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "gradient_boosting": "Gradient Boosting",
    "xgboost": "XGBoost",
    "linear_svm": "Linear SVM",
    "gaussian_naive_bayes": "Gaussian Naive Bayes",
}
MODEL_FAMILY = {
    "logistic_regression": "linear probabilistic",
    "decision_tree": "single tree",
    "random_forest": "bagged tree ensemble",
    "gradient_boosting": "sequential boosting ensemble",
    "xgboost": "regularized boosting ensemble",
    "linear_svm": "maximum-margin linear",
    "gaussian_naive_bayes": "probabilistic conditional-independence",
}
PROVENANCE = {
    "logistic_regression": "P01;P02;P03",
    "decision_tree": "P02;P03",
    "random_forest": "P02;P03",
    "gradient_boosting": "P03;Friedman2001",
    "xgboost": "P03;P05;ChenGuestrin2016",
    "linear_svm": "P02;P03;CortesVapnik1995",
    "gaussian_naive_bayes": "P02;FriedmanGeigerGoldszmidt1997",
}
THRESHOLD = {
    "logistic_regression": 0.5,
    "decision_tree": 0.5,
    "random_forest": 0.5,
    "gradient_boosting": 0.5,
    "xgboost": 0.5,
    "linear_svm": 0.0,
    "gaussian_naive_bayes": 0.5,
}

scores = {
    "logistic_regression": np.asarray(y_test_prob),
    "decision_tree": np.asarray(y_test_prob_dt),
    "random_forest": np.asarray(y_test_prob_rf),
    "gradient_boosting": np.asarray(y_test_prob_gb),
    "xgboost": np.asarray(y_test_prob_xgb),
    "linear_svm": np.asarray(y_test_score_svm),
    "gaussian_naive_bayes": np.asarray(y_test_prob_gnb),
}
preds = {
    "logistic_regression": np.asarray(y_test_pred),
    "decision_tree": np.asarray(y_test_pred_dt),
    "random_forest": np.asarray(y_test_pred_rf),
    "gradient_boosting": np.asarray(y_test_pred_gb),
    "xgboost": np.asarray(y_test_pred_xgb),
    "linear_svm": np.asarray(y_test_pred_svm),
    "gaussian_naive_bayes": np.asarray(y_test_pred_gnb),
}
cv_tables = {
    "logistic_regression": logreg_cv_summary,
    "decision_tree": dt_cv_summary,
    "random_forest": rf_cv_summary,
    "gradient_boosting": gb_cv_summary,
    "xgboost": xgb_cv_summary,
    "linear_svm": svm_cv_summary,
    "gaussian_naive_bayes": gnb_cv_summary,
}
test_ci_tables = {
    "logistic_regression": logreg_test_summary,
    "decision_tree": dt_test_summary,
    "random_forest": rf_test_summary,
    "gradient_boosting": gb_test_summary,
    "xgboost": xgb_test_summary,
    "linear_svm": svm_test_summary,
    "gaussian_naive_bayes": gnb_test_summary,
}

# Previously verified frozen point estimates. These are a guardrail, not a new tuning target.
FROZEN = {
    "logistic_regression": {"accuracy":0.8558270566941087,"f1":0.2831845562087234,"roc_auc":0.8261678122652486,"pr_auc":0.44663290841217973},
    "decision_tree": {"accuracy":0.78615636130087,"f1":0.32351311414850387,"roc_auc":0.6020699144719784,"pr_auc":0.20533779176339034},
    "random_forest": {"accuracy":0.8510392946809132,"f1":0.2773174704341052,"roc_auc":0.8044953607113304,"pr_auc":0.40246501462675927},
    "gradient_boosting": {"accuracy":0.8563525427687277,"f1":0.26630880489077763,"roc_auc":0.8260155801805384,"pr_auc":0.4483169949856431},
    "xgboost": {"accuracy":0.8548198750510889,"f1":0.2904836638607505,"roc_auc":0.8262898793557036,"pr_auc":0.44171377834099224},
    "linear_svm": {"accuracy":0.8559876218835757,"f1":0.20242522231204527,"roc_auc":0.8261580968770885,"pr_auc":0.44788405460506064},
    "gaussian_naive_bayes": {"accuracy":0.7272873241081333,"f1":0.4400083925306477,"roc_auc":0.7885646728624294,"pr_auc":0.36412201184388493},
}

y_compare = y_test.to_numpy()
paired_predictions = pd.DataFrame({
    "test_row_id": X_test.index.to_numpy(),
    "y_true": y_compare,
})
for name in MODEL_ORDER:
    paired_predictions[f"{name}_score"] = scores[name]
    paired_predictions[f"{name}_pred"] = preds[name]

paired_predictions.to_csv(
    COMPARISON_DIR / "paired_test_predictions.csv.gz",
    index=False,
    compression="gzip",
)

def metric_point(name):
    pred = preds[name]
    score = scores[name]
    tn, fp, fn, tp = confusion_matrix(y_compare, pred, labels=[0,1]).ravel()
    row = {
        "accuracy": accuracy_score(y_compare, pred),
        "balanced_accuracy": balanced_accuracy_score(y_compare, pred),
        "precision": precision_score(y_compare, pred, zero_division=0),
        "recall": recall_score(y_compare, pred),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y_compare, pred),
        "roc_auc": roc_auc_score(y_compare, score),
        "pr_auc": average_precision_score(y_compare, score),
        "brier_score": np.nan if name == "linear_svm" else brier_score_loss(y_compare, score),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    return row

point = {name: metric_point(name) for name in MODEL_ORDER}

for name in MODEL_ORDER:
    for metric, expected in FROZEN[name].items():
        delta = abs(point[name][metric] - expected)
        if delta > 1e-6:
            raise RuntimeError(
                f"Frozen-result drift: {name} {metric} current={point[name][metric]:.12f} "
                f"expected={expected:.12f} abs_delta={delta:.3g}"
            )

print("Frozen reconciliation passed for all seven models.")


## 4.1 Canonical seven-model table

Each row combines:

- model family and literature provenance;
- **CV mean ± SD** (descriptive; folds are correlated);
- held-out test point estimates;
- held-out stratified-bootstrap 95% intervals;
- confusion-matrix counts;
- Brier score only where a valid probability is available.

Balanced Accuracy is included because the positive class is only about 15% of the modeling cohort.


In [ ]:
comparison_rows = []
metric_names = ["accuracy","balanced_accuracy","precision","recall","specificity","f1","roc_auc","pr_auc"]

for name in MODEL_ORDER:
    row = {
        "model_id": name,
        "model": DISPLAY_NAMES[name],
        "model_family": MODEL_FAMILY[name],
        "provenance": PROVENANCE[name],
        "decision_threshold": THRESHOLD[name],
    }

    cv_table = cv_tables[name].set_index("metric")
    test_table = test_ci_tables[name].set_index("metric")

    for metric in metric_names:
        row[f"cv_{metric}_mean"] = float(cv_table.loc[metric, "mean"])
        row[f"cv_{metric}_std"] = float(cv_table.loc[metric, "std"])
        row[f"cv_{metric}_min"] = float(cv_table.loc[metric, "min"])
        row[f"cv_{metric}_max"] = float(cv_table.loc[metric, "max"])

        row[f"test_{metric}"] = float(point[name][metric])
        row[f"test_{metric}_ci95_lower"] = float(test_table.loc[metric, "ci95_lower"])
        row[f"test_{metric}_ci95_upper"] = float(test_table.loc[metric, "ci95_upper"])

    row["test_brier_score"] = point[name]["brier_score"]
    if name != "linear_svm" and "brier_score" in test_table.index:
        row["test_brier_ci95_lower"] = float(test_table.loc["brier_score", "ci95_lower"])
        row["test_brier_ci95_upper"] = float(test_table.loc["brier_score", "ci95_upper"])
    else:
        row["test_brier_ci95_lower"] = np.nan
        row["test_brier_ci95_upper"] = np.nan

    for k in ["tn","fp","fn","tp"]:
        row[k] = int(point[name][k])

    comparison_rows.append(row)

model_comparison = pd.DataFrame(comparison_rows)
model_comparison.to_csv(COMPARISON_DIR / "model_comparison.csv", index=False)

display_cols = [
    "model","test_accuracy","test_balanced_accuracy","test_precision","test_recall","test_specificity",
    "test_f1","test_roc_auc","test_pr_auc","test_brier_score"
]
display(model_comparison[display_cols].sort_values("test_roc_auc", ascending=False))


## 4.2 Paired stratified bootstrap

**Prespecified reference:** Logistic Regression is the transparent baseline.

For every frozen bootstrap replicate:
1. positive and negative held-out respondents are resampled separately;
2. the **same respondent indices are applied to every model**;
3. model-minus-model differences are computed from the frozen prediction vectors.

The primary notebook uses 200 paired replicates. These intervals are **estimation-focused and unadjusted for multiplicity**; they are not equivalence tests and are not treated as confirmatory p-values.

A pre-submission robustness audit re-used the exact same frozen respondent-level predictions with **5,000 paired stratified replicates** for the leading LR/Gradient-Boosting/XGBoost/Linear-SVM discrimination contrasts. ROC-AUC conclusions were unchanged. The small LR–XGBoost AP contrast sits near the inferential boundary and is therefore reported as a small model-dependent difference rather than evidence of overall superiority.

All non-baseline all-pairs bootstrap contrasts remain secondary/exploratory.


In [ ]:
PAIRED_BOOT_REPS = 200
PAIRED_BOOT_SEED = 20251010
paired_rng = np.random.default_rng(PAIRED_BOOT_SEED)

pos_compare = np.flatnonzero(y_compare == 1)
neg_compare = np.flatnonzero(y_compare == 0)
paired_metric_names = ["accuracy","balanced_accuracy","precision","recall","specificity","f1","roc_auc","pr_auc"]

boot_records = []
for b in range(PAIRED_BOOT_REPS):
    s_pos = paired_rng.choice(pos_compare, size=len(pos_compare), replace=True)
    s_neg = paired_rng.choice(neg_compare, size=len(neg_compare), replace=True)
    idx = np.concatenate([s_pos, s_neg])
    paired_rng.shuffle(idx)
    yb = y_compare[idx]

    for name in MODEL_ORDER:
        pb = preds[name][idx]
        sb = scores[name][idx]
        tn, fp, fn, tp = confusion_matrix(yb, pb, labels=[0,1]).ravel()
        rec = {
            "replicate": b + 1,
            "model": name,
            "accuracy": accuracy_score(yb, pb),
            "balanced_accuracy": balanced_accuracy_score(yb, pb),
            "precision": precision_score(yb, pb, zero_division=0),
            "recall": recall_score(yb, pb),
            "specificity": tn / (tn + fp),
            "f1": f1_score(yb, pb),
            "roc_auc": roc_auc_score(yb, sb),
            "pr_auc": average_precision_score(yb, sb),
            "brier_score": np.nan if name == "linear_svm" else brier_score_loss(yb, sb),
        }
        boot_records.append(rec)

paired_boot = pd.DataFrame(boot_records)
paired_boot.to_csv(COMPARISON_DIR / "paired_bootstrap_model_metrics.csv", index=False)

pair_rows = []
for a, b in combinations(MODEL_ORDER, 2):
    a_df = paired_boot[paired_boot.model == a].sort_values("replicate")
    b_df = paired_boot[paired_boot.model == b].sort_values("replicate")

    metrics = paired_metric_names + (
        ["brier_score"] if a != "linear_svm" and b != "linear_svm" else []
    )
    for metric in metrics:
        diff = a_df[metric].to_numpy() - b_df[metric].to_numpy()
        pair_rows.append({
            "model_a": a,
            "model_b": b,
            "metric": metric,
            "point_difference_a_minus_b": point[a][metric] - point[b][metric],
            "bootstrap_mean_difference": float(diff.mean()),
            "ci95_lower": float(np.quantile(diff, 0.025)),
            "ci95_upper": float(np.quantile(diff, 0.975)),
            "ci_excludes_zero_unadjusted": bool(
                np.quantile(diff, 0.025) > 0 or np.quantile(diff, 0.975) < 0
            ),
        })

paired_differences = pd.DataFrame(pair_rows)
paired_differences.to_csv(COMPARISON_DIR / "paired_bootstrap_differences.csv", index=False)

primary_vs_logistic = paired_differences[
    (paired_differences["model_a"] == "logistic_regression")
    & paired_differences["metric"].isin(["roc_auc", "pr_auc", "balanced_accuracy", "f1"])
]
display(primary_vs_logistic)


## 4.3 McNemar Tests at the Frozen Decision Boundaries

McNemar's exact test evaluates whether two classifiers have different error rates on the same respondents.

All 21 model pairs are tested. Because this is a family of multiple tests, the notebook applies a **Holm step-down family-wise error correction**.

These tests describe disagreement at each model's frozen/native decision boundary; they do not replace ROC-AUC, Average Precision (AP), calibration, or the paired bootstrap analysis.


In [ ]:
correct = {name: preds[name] == y_compare for name in MODEL_ORDER}

mcnemar_rows = []
for a, b in combinations(MODEL_ORDER, 2):
    a_correct_b_wrong = int(np.sum(correct[a] & ~correct[b]))
    a_wrong_b_correct = int(np.sum(~correct[a] & correct[b]))
    discordant = a_correct_b_wrong + a_wrong_b_correct

    p = 1.0 if discordant == 0 else float(
        binomtest(
            k=min(a_correct_b_wrong, a_wrong_b_correct),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue
    )

    mcnemar_rows.append({
        "model_a": a,
        "model_b": b,
        "a_correct_b_wrong": a_correct_b_wrong,
        "a_wrong_b_correct": a_wrong_b_correct,
        "discordant_n": discordant,
        "accuracy_difference_a_minus_b": point[a]["accuracy"] - point[b]["accuracy"],
        "p_value_exact": p,
    })

mcnemar = pd.DataFrame(mcnemar_rows).sort_values("p_value_exact").reset_index(drop=True)

m = len(mcnemar)
raw = mcnemar["p_value_exact"].to_numpy()
adjusted = np.maximum.accumulate(np.minimum(1.0, (m - np.arange(m)) * raw))
mcnemar["p_value_holm"] = adjusted
mcnemar["significant_holm_0_05"] = mcnemar["p_value_holm"] < 0.05

mcnemar.to_csv(COMPARISON_DIR / "mcnemar_pairwise_holm.csv", index=False)
display(mcnemar)


## 4.4 Combined ROC and Precision–Recall Curves

These figures use the **same held-out predictions already produced by Models 01–07**. No model is refit and no threshold is tuned.


In [ ]:
plt.figure(figsize=(9,7))
for name in MODEL_ORDER:
    fpr_c, tpr_c, _ = roc_curve(y_compare, scores[name])
    auc = roc_auc_score(y_compare, scores[name])
    plt.plot(fpr_c, tpr_c, label=f"{DISPLAY_NAMES[name]} ({auc:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("BRFSS 2025 — Seven Frozen Models: ROC Comparison")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / "roc_comparison.png", dpi=180)
plt.show()

plt.figure(figsize=(9,7))
for name in MODEL_ORDER:
    p_curve, r_curve, _ = precision_recall_curve(y_compare, scores[name])
    ap = average_precision_score(y_compare, scores[name])
    plt.plot(r_curve, p_curve, label=f"{DISPLAY_NAMES[name]} ({ap:.3f})")
plt.axhline(y=y_compare.mean(), linestyle="--", label=f"Sample prevalence ({y_compare.mean():.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("BRFSS 2025 — Seven Frozen Models: Precision–Recall Comparison")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / "pr_comparison.png", dpi=180)
plt.show()


# Compact metric matrix for fast model comparison.
matrix_metrics = [
    "test_accuracy", "test_balanced_accuracy", "test_precision", "test_recall",
    "test_specificity", "test_f1", "test_roc_auc", "test_pr_auc",
]
matrix = model_comparison.set_index("model")[matrix_metrics]
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(matrix.to_numpy(), aspect="auto")
ax.set_xticks(range(len(matrix_metrics)))
ax.set_xticklabels([m.replace("test_", "") for m in matrix_metrics], rotation=35, ha="right")
ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f"{matrix.iloc[i, j]:.3f}", ha="center", va="center", fontsize=8)
ax.set_title("Seven-Model Held-Out Metric Matrix")
fig.colorbar(im, ax=ax, label="Metric value")
fig.tight_layout()
fig.savefig(COMPARISON_DIR / "metric_matrix.png", dpi=180)
plt.show()

# Operating-point trade-off.
plt.figure(figsize=(7, 6))
for _, row in model_comparison.iterrows():
    plt.scatter(row["test_recall"], row["test_precision"])
    plt.annotate(row["model"], (row["test_recall"], row["test_precision"]), fontsize=8)
plt.xlabel("Recall / Sensitivity")
plt.ylabel("Precision / PPV")
plt.title("Frozen Decision-Boundary Trade-off")
plt.tight_layout()
plt.savefig(COMPARISON_DIR / "precision_recall_tradeoff.png", dpi=180)
plt.show()

# Calibration is meaningful only for probability-producing models.
from sklearn.calibration import calibration_curve

plt.figure(figsize=(7, 6))
for name in MODEL_ORDER:
    if name == "linear_svm":
        continue
    frac_pos, mean_pred = calibration_curve(
        y_compare, scores[name], n_bins=10, strategy="quantile"
    )
    plt.plot(mean_pred, frac_pos, marker="o", label=DISPLAY_NAMES[name])
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed positive fraction")
plt.title("Probability Calibration — 10 Quantile Bins")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / "calibration_comparison.png", dpi=180)
plt.show()

# Aggregate encoded-column diagnostics back to the 20 original BRFSS variables.
fitted_pipelines = {
    "logistic_regression": logreg_pipeline,
    "decision_tree": decision_tree_pipeline,
    "random_forest": random_forest_pipeline,
    "gradient_boosting": gb_pipeline,
    "xgboost": xgb_pipeline,
    "linear_svm": svm_pipeline,
}

importance_rows = []
for name, pipe in fitted_pipelines.items():
    prep = pipe.named_steps["preprocess"]
    onehot = prep.named_transformers_["categorical"].named_steps["onehot"]
    source_feature = []
    for feature, categories in zip(CATEGORICAL_FEATURES, onehot.categories_):
        source_feature.extend([feature] * len(categories))
    source_feature.extend(NUMERIC_FEATURES)

    model = pipe.named_steps["model"]
    if hasattr(model, "feature_importances_"):
        raw_importance = np.asarray(model.feature_importances_)
        diagnostic = "tree_importance"
    else:
        raw_importance = np.abs(np.asarray(model.coef_).ravel())
        diagnostic = "abs_coefficient"

    if len(raw_importance) != len(source_feature):
        raise RuntimeError(
            f"Feature-diagnostic dimension mismatch for {name}: "
            f"{len(raw_importance)} vs {len(source_feature)}"
        )

    tmp = pd.DataFrame({
        "source_feature": source_feature,
        "raw_importance": raw_importance,
    }).groupby("source_feature", as_index=False)["raw_importance"].sum()

    total = tmp["raw_importance"].sum()
    tmp["relative_importance"] = tmp["raw_importance"] / total if total > 0 else 0.0
    tmp["model_id"] = name
    tmp["model"] = DISPLAY_NAMES[name]
    tmp["diagnostic_type"] = diagnostic
    importance_rows.append(tmp)

feature_diagnostics = pd.concat(importance_rows, ignore_index=True)
feature_diagnostics.to_csv(
    COMPARISON_DIR / "aggregated_feature_diagnostics.csv", index=False
)

importance_pivot = feature_diagnostics.pivot(
    index="source_feature", columns="model", values="relative_importance"
).fillna(0.0)
fig, ax = plt.subplots(figsize=(9, 9))
im = ax.imshow(importance_pivot.to_numpy(), aspect="auto")
ax.set_xticks(range(len(importance_pivot.columns)))
ax.set_xticklabels(importance_pivot.columns, rotation=35, ha="right")
ax.set_yticks(range(len(importance_pivot.index)))
ax.set_yticklabels(importance_pivot.index)
ax.set_title("Model-Specific Feature Diagnostics Aggregated to Original BRFSS Variables")
fig.colorbar(im, ax=ax, label="Within-model relative diagnostic")
fig.tight_layout()
fig.savefig(COMPARISON_DIR / "feature_diagnostic_matrix.png", dpi=180)
plt.show()


# Part V — Interpretation, Limitations, and Reporting Rules

## 5.1 How to read the benchmark

| Question | Preferred evidence |
|---|---|
| Ranking / discrimination | ROC-AUC + Average Precision (AP) |
| Imbalanced-class operating performance | Balanced Accuracy, Recall, Precision, F1 |
| False-positive control | Specificity + Precision |
| Probability quality | Brier score + calibration curve |
| Model-to-model uncertainty | Paired bootstrap effect estimates |
| Thresholded error disagreement | Holm-corrected McNemar |

A single metric does **not** define a universal winner. A confidence interval crossing zero means that this analysis did not clearly distinguish the models on that contrast; it does **not** prove statistical equivalence.

### Naive reference

On the held-out test set, an always-negative classifier would achieve approximately **0.8487 accuracy** simply because the positive class is 15.13%, while Balanced Accuracy and ROC-AUC would be 0.50 and F1 would be 0. This is why raw accuracy is never interpreted alone.

## 5.2 Limitations that must remain visible

| Limitation | Consequence |
|---|---|
| Cross-sectional self-report outcome | No prospective-incidence claim; no laboratory-confirmed diagnosis |
| Concurrent comorbidities | Classification association only; temporal causality unknown |
| Unweighted ML benchmark | 15.13% is a sample proportion, not national prevalence |
| 2025 public-file coverage | Not all U.S. jurisdictions are represented in the aggregate release |
| Random internal holdout | No temporal, geographic, or external validation |
| Fixed untuned configurations | Comparison is between prespecified baselines, not each algorithm family's optimum |
| Frozen/default decision boundaries | Recall/precision differences partly reflect operating point |
| Missing-data assumptions | Explicit-category + median imputation is pragmatic, not assumption-free |
| Primary bootstrap uses 200 replicates | Adequate for coarse uncertainty summaries but borderline tiny contrasts require higher-repetition sensitivity checks |
| Primary calibration reporting | Brier + calibration curves; calibration intercept/slope are provided in the pre-submission supplement, not used for recalibration |
| No clinical decision-curve analysis | Clinical net benefit is not claimed because no intended intervention threshold was specified |
| No subgroup/fairness performance analysis in the frozen primary benchmark | No fairness/deployment claim is made; subgroup validation is required before any operational use |
| CDC 284 vs pandas 283 columns | Parser/schema discrepancy remains documented; every target/predictor used here is explicitly verified as present |

## 5.3 Claim discipline

Use **“classification of self-reported diagnosed diabetes status”** and **“internal validation”**.

Use **“Average Precision (AP)”** for values computed by `average_precision_score`; the legacy artifact key `pr_auc` is retained only for backward compatibility.

Do not:
- describe the study as external validation or a population-representative prevalence analysis;
- infer causality from feature importance or coefficients;
- claim statistical equivalence without a prespecified equivalence margin/test;
- claim clinical utility, fairness, or deployment readiness from this benchmark;
- select or retune a model/threshold using the held-out test outcomes.


In [ ]:
leaders = {
    "balanced_accuracy": model_comparison.loc[model_comparison["test_balanced_accuracy"].idxmax(), ["model","test_balanced_accuracy"]].to_dict(),
    "accuracy": model_comparison.loc[model_comparison["test_accuracy"].idxmax(), ["model","test_accuracy"]].to_dict(),
    "precision": model_comparison.loc[model_comparison["test_precision"].idxmax(), ["model","test_precision"]].to_dict(),
    "recall": model_comparison.loc[model_comparison["test_recall"].idxmax(), ["model","test_recall"]].to_dict(),
    "specificity": model_comparison.loc[model_comparison["test_specificity"].idxmax(), ["model","test_specificity"]].to_dict(),
    "f1": model_comparison.loc[model_comparison["test_f1"].idxmax(), ["model","test_f1"]].to_dict(),
    "roc_auc": model_comparison.loc[model_comparison["test_roc_auc"].idxmax(), ["model","test_roc_auc"]].to_dict(),
    "pr_auc": model_comparison.loc[model_comparison["test_pr_auc"].idxmax(), ["model","test_pr_auc"]].to_dict(),
}

probability_models = model_comparison[model_comparison["test_brier_score"].notna()]
best_brier = probability_models.loc[
    probability_models["test_brier_score"].idxmin(),
    ["model","test_brier_score"]
].to_dict()

top4_names = ["logistic_regression","gradient_boosting","xgboost","linear_svm"]
top4_auc = model_comparison[model_comparison.model_id.isin(top4_names)]["test_roc_auc"]
top4_auc_range = float(top4_auc.max() - top4_auc.min())

print("Point-estimate leaders:")
for metric, result in leaders.items():
    print(f"  {metric:12s}: {result['model']} ({result['test_'+metric]:.6f})")
print(f"  brier_score : {best_brier['model']} ({best_brier['test_brier_score']:.6f})")
print()
print(f"ROC-AUC spread among LR / Gradient Boosting / XGBoost / Linear SVM: {top4_auc_range:.6f}")
print("Use paired_bootstrap_differences.csv before interpreting that small spread as meaningful.")

manifest = {
    "study": "BRFSS 2025 Diabetes Classification",
    "canonical_notebook": "BRFSS_2025_Diabetes_Classification.ipynb",
    "single_notebook_pipeline": True,
    "primary_models": MODEL_ORDER,
    "test_n": int(len(y_compare)),
    "paired_bootstrap_reps": PAIRED_BOOT_REPS,
    "paired_bootstrap_seed": PAIRED_BOOT_SEED,
    "mcnemar_multiple_comparison_correction": "Holm FWER",
    "model_retuning_after_test": False,
    "threshold_optimization_on_test": False,
    "linear_svm_brier_score": None,
    "python": platform.python_version(),
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
}
with open(COMPARISON_DIR / "comparison_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
